# Notebook 07 — พอร์ตโฟลิโอแบบจักรวาลหุ้นเปลี่ยนแปลงได้

สมุดงานนี้กำหนดโครงสร้างตรวจสอบข้อมูลและตัวประมวลผลสำหรับการวิจัยเท่านั้น ไม่ใช่หลักฐานว่า backtest ผ่านเกณฑ์ ไม่เรียกข้อมูลภายนอก ไม่ฝึกหรือทำนายด้วยโมเดล และไม่รันผลย้อนหลังในสถานะปัจจุบัน

**สถานะคงไว้:** `RESEARCH_BACKTEST_GATE = BLOCKED` และ `PHASE_F = DO_NOT_BEGIN`

## ส่วนที่ 01 — ขอบเขตข้อมูลและการตรวจสอบความครบถ้วน

อ่าน source และ metadata ในเครื่องแบบ read-only พร้อมคำนวณ SHA-256 จากไฟล์จริง ไฟล์ Master เป็นเพียง candidate ไม่ใช่หลักฐานยืนยัน training table หรือ Feature Snapshot โมเดลถูกโหลดเพื่ออ่าน metadata เท่านั้น ส่วน Execution Snapshot เป็นข้อมูลจาก Yahoo Finance และยังไม่ได้รับอนุมัติเป็นราคาซื้อขายจากตลาดหลักทรัพย์

การตรวจนี้ไม่ใช้ `fit`, `predict`, `predict_proba`, API หรือการดาวน์โหลดใด ๆ

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import platform
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Iterable, Mapping, Sequence

import joblib
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.optimize import linprog, minimize
from scipy.spatial.distance import squareform

FEATURES = [
    "daily_return",
    "momentum_20d",
    "vol_z",
    "VolRank",
    "CorrRank",
    "LiquidityRank",
]
STRATEGIES = [
    "S1_PanicBuy_FOMOBuy",
    "S2_PanicBuy_FOMOSell",
    "S3_PanicBuy_FOMOHold",
    "S4_PanicSell_FOMOBuy",
    "S5_PanicSell_FOMOSell",
    "S6_PanicSell_FOMOHold",
    "S7_PanicHold_FOMOBuy",
    "S8_PanicHold_FOMOSell",
    "S9_PanicHold_FOMOHold",
]
STRATEGY_ACTION_MATRIX = {
    "S1_PanicBuy_FOMOBuy": {0: "BUY", 1: "BUY", 2: "HOLD"},
    "S2_PanicBuy_FOMOSell": {0: "BUY", 1: "SELL", 2: "HOLD"},
    "S3_PanicBuy_FOMOHold": {0: "BUY", 1: "HOLD", 2: "HOLD"},
    "S4_PanicSell_FOMOBuy": {0: "SELL", 1: "BUY", 2: "HOLD"},
    "S5_PanicSell_FOMOSell": {0: "SELL", 1: "SELL", 2: "HOLD"},
    "S6_PanicSell_FOMOHold": {0: "SELL", 1: "HOLD", 2: "HOLD"},
    "S7_PanicHold_FOMOBuy": {0: "HOLD", 1: "BUY", 2: "HOLD"},
    "S8_PanicHold_FOMOSell": {0: "HOLD", 1: "SELL", 2: "HOLD"},
    "S9_PanicHold_FOMOHold": {0: "HOLD", 1: "HOLD", 2: "HOLD"},
}
ALLOCATION_METHODS = ["EW", "IV", "ERC", "HRP", "MinVar", "CVaR", "MPT"]
E029_LINEAGE_FIELDS = [
    "Universe_Snapshot_ID",
    "Prediction_Snapshot_ID",
    "Execution_Eligibility_Snapshot_ID",
    "Model_Version",
    "Allocation_Policy_Version",
]
EXPECTED_EXECUTION_SHA256 = (
    "14ca82c9a481118910952cfaaaeea08daecd6548d9d99e19a66e4d9c51ef986a"
)
BOARD_LOT_DEFAULT = 100
MAX_POSITION_WEIGHT = 0.25
MAX_SLIPPAGE_RATE = 0.001
TRANSACTION_COST_RATE = 0.0016799
ANNUALIZATION_INTERVALS = 252
MIN_ANNUALIZED_MARKS = 253
MAX_STALE_VERIFIED_SESSIONS = 3
ERC_MAX_RELATIVE_RC_DEVIATION = 0.05
ERC_MAX_CONDITION_NUMBER = 1e12


class PolicyBlockedError(RuntimeError):
    def __init__(self, reason: str, detail: str = "") -> None:
        self.reason = reason
        self.detail = detail
        super().__init__(f"{reason}: {detail}" if detail else reason)


class NoLookaheadError(PolicyBlockedError):
    pass


def locate_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "backtest_results").is_dir() and (candidate / "dataset").is_dir():
            return candidate
    raise FileNotFoundError("Cannot locate project root from the current directory.")


PROJECT_ROOT = locate_project_root()
OUTPUT_DIR = PROJECT_ROOT / "backtest_results" / "notebook_07_clean_implementation"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_frame_hash(frame: pd.DataFrame) -> str:
    canonical = frame.copy()
    canonical.index.name = canonical.index.name or "row"
    payload = canonical.to_csv(
        index=True, lineterminator="\n", float_format="%.17g"
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def finite_positive(value: Any) -> bool:
    try:
        return bool(np.isfinite(float(value)) and float(value) > 0)
    except (TypeError, ValueError):
        return False


def require_unique_key(frame: pd.DataFrame, keys: Sequence[str], label: str) -> None:
    missing = [key for key in keys if key not in frame.columns]
    if missing:
        raise PolicyBlockedError("SCHEMA_INVALID", f"{label} missing {missing}")
    if frame.duplicated(list(keys)).any():
        raise PolicyBlockedError("DUPLICATE_KEY", f"{label} has duplicate {list(keys)}")


In [2]:
def inspect_local_sources() -> dict[str, Any]:
    dataset_dir = PROJECT_ROOT / "dataset" / "verified_build_20261113_152948"
    model_dir = PROJECT_ROOT / "models" / "Original_Softmax_Final"
    phase_c_dir = PROJECT_ROOT / "backtest_results" / "phase_C_vendor_validation_20261116"
    phase_e_dir = PROJECT_ROOT / "backtest_results" / "phase_E_portfolio_specification"
    paths = {
        "Feature_Candidate": dataset_dir
        / "SET50_Master_Dataset_DateSymbol_Sorted_VolZ_Fixed.parquet",
        "Execution_Snapshot": dataset_dir
        / "SET50_Execution_OHLCV_2026_Yahoo_auto_adjust_false_snapshot.parquet",
        "Execution_Sidecar": dataset_dir
        / "SET50_Execution_OHLCV_2026_Yahoo_auto_adjust_false_snapshot.json",
        "Model": model_dir / "final_softmax_model.joblib",
        "Model_Metadata": model_dir / "final_softmax_metadata.json",
        "Membership_Periods": PROJECT_ROOT
        / "dataset"
        / "SET50_constituents_by_halfyear_2016_2026.csv",
        "Membership_Changes": PROJECT_ROOT / "dataset" / "SET50_membership_changes.csv",
        "Membership_2026_Evidence": phase_c_dir / "membership_2026_primary_evidence_audit.csv",
        "Exclusions": PROJECT_ROOT
        / "backtest_results"
        / "research_controls"
        / "execution_excluded_symbols.csv",
        "Corporate_Actions": PROJECT_ROOT
        / "backtest_results"
        / "corporate_action_ledger_official.csv",
        "Decision_Register": phase_e_dir / "phase_E_decision_register.csv",
    }
    audit_rows: list[dict[str, Any]] = []

    def add_file_record(name: str, path: Path, status: str, details: str = "") -> None:
        if not path.is_file():
            raise FileNotFoundError(f"Required audit input is missing: {path}")
        stat = path.stat()
        audit_rows.append(
            {
                "Artifact": name,
                "Full_Path": str(path),
                "File_Size_Bytes": stat.st_size,
                "Modified_Local": pd.Timestamp(stat.st_mtime, unit="s").isoformat(),
                "SHA256": sha256_file(path),
                "Status": status,
                "Details": details,
            }
        )

    for key in ("Feature_Candidate", "Execution_Snapshot", "Model", "Model_Metadata"):
        add_file_record(key, paths[key], "HASHED_READ_ONLY")
    for key in (
        "Membership_Periods",
        "Membership_Changes",
        "Membership_2026_Evidence",
        "Exclusions",
        "Corporate_Actions",
        "Decision_Register",
    ):
        add_file_record(key, paths[key], "HASHED_READ_ONLY")

    execution_sidecar = json.loads(paths["Execution_Sidecar"].read_text(encoding="utf-8"))
    add_file_record(
        "Execution_Sidecar",
        paths["Execution_Sidecar"],
        "METADATA_READ_ONLY",
        f"source={execution_sidecar.get('source')}; retrieved={execution_sidecar.get('downloaded_at_utc')}",
    )
    execution_hash = next(
        row["SHA256"] for row in audit_rows if row["Artifact"] == "Execution_Snapshot"
    )
    execution_hash_matches = execution_hash == EXPECTED_EXECUTION_SHA256
    execution = pd.read_parquet(paths["Execution_Snapshot"])
    master_candidate = pd.read_parquet(paths["Feature_Candidate"])
    membership_periods = pd.read_csv(paths["Membership_Periods"])
    membership_changes = pd.read_csv(paths["Membership_Changes"])
    membership_2026 = pd.read_csv(paths["Membership_2026_Evidence"])
    exclusions = pd.read_csv(paths["Exclusions"])
    corporate_actions = pd.read_csv(paths["Corporate_Actions"])
    decision_register = pd.read_csv(paths["Decision_Register"])
    metadata = json.loads(paths["Model_Metadata"].read_text(encoding="utf-8"))
    model = joblib.load(paths["Model"])

    feature_names = list(getattr(model, "feature_names_in_", []))
    model_classes = [int(value) for value in getattr(model, "classes_", [])]
    metadata_features = list(metadata.get("features", []))
    model_metadata_match = feature_names == FEATURES and metadata_features == FEATURES
    missing_candidate_features = sorted(set(FEATURES).difference(master_candidate.columns))
    execution_required = {"Date", "Symbol", "Open", "High", "Low", "Close", "Volume"}
    missing_execution_columns = sorted(execution_required.difference(execution.columns))
    if missing_execution_columns:
        raise PolicyBlockedError(
            "EXECUTION_SCHEMA_INVALID", str(missing_execution_columns)
        )
    if "Date" not in master_candidate or "Symbol" not in master_candidate:
        raise PolicyBlockedError("FEATURE_CANDIDATE_SCHEMA_INVALID")
    require_unique_key(execution, ["Date", "Symbol"], "Execution snapshot")
    require_unique_key(master_candidate, ["Date", "Symbol"], "Master candidate")
    membership_has_effective_date = "Effective_Date" in membership_changes.columns
    verified_rows = membership_2026[
        membership_2026["Status"].eq("VERIFIED_FROM_OFFICIAL_SET_PDF")
        & membership_2026["Local_Official_Match"].astype(str).str.lower().eq("true")
    ]
    verified_membership_counts = verified_rows.groupby("Period")["Symbol"].nunique().to_dict()
    exclusions_approved = exclusions[
        exclusions["Decision_Status"].eq("APPROVED_RESEARCH_EXCLUSION")
    ]["Symbol"].astype(str).tolist()
    decision_statuses = (
        decision_register.set_index("Decision_ID")["Current_Status"].to_dict()
    )
    decision_status_checks = {
        "E-001_BLOCKED": decision_statuses.get("E-001") == "BLOCKED",
        "E-002_NEEDS_APPROVAL": decision_statuses.get("E-002")
        == "NEEDS_RESEARCHER_APPROVAL",
        "E-004_APPROVED": decision_statuses.get("E-004") == "APPROVED",
        "E-005_TO_E-020_APPROVED": all(
            decision_statuses.get(f"E-{number:03d}") == "APPROVED"
            for number in range(5, 21)
        ),
        "E-021_E-022_BLOCKED": decision_statuses.get("E-021") == "BLOCKED"
        and decision_statuses.get("E-022") == "BLOCKED",
        "E-023_VERIFIED": decision_statuses.get("E-023") == "VERIFIED",
        "E-027_E-028_APPROVED": decision_statuses.get("E-027") == "APPROVED"
        and decision_statuses.get("E-028") == "APPROVED",
        "E-029_APPROVED": decision_statuses.get("E-029") == "APPROVED",
    }
    ledger_statuses = (
        corporate_actions["Reconciliation_Status"].dropna().astype(str).unique().tolist()
        if "Reconciliation_Status" in corporate_actions
        else []
    )
    source_is_vendor = "yahoo" in str(execution_sidecar.get("source", "")).lower()
    inventory = pd.DataFrame(audit_rows)
    readiness = {
        "feature_source_status": "UNKNOWN_INSUFFICIENT_EVIDENCE",
        "feature_candidate_status": "CANDIDATE_NOT_ATTRIBUTED_TO_TRAINING",
        "feature_candidate_missing_approved_columns": missing_candidate_features,
        "model_artifact_feature_names": feature_names,
        "model_metadata_features": metadata_features,
        "model_metadata_matches_artifact": model_metadata_match,
        "model_artifact_classes": model_classes,
        "model_inference_allowed": False,
        "execution_hash_matches_expected": execution_hash_matches,
        "execution_source_is_vendor": source_is_vendor,
        "execution_is_official_set_prints": False,
        "execution_status": "VENDOR_NOT_APPROVED_AS_EXECUTION_PRICES",
        "membership_period_file_has_effective_date": membership_has_effective_date,
        "membership_verified_counts_by_period": verified_membership_counts,
        "membership_status": "VERIFIED_2026_H1_H2_ONLY",
        "exclusion_symbols": exclusions_approved,
        "corporate_action_reconciliation_statuses": ledger_statuses,
        "corporate_action_status": "BLOCKED_PENDING_OFFICIAL_RECONCILIATION",
        "calendar_status": "BLOCKED_NO_VERIFIED_OFFICIAL_SESSION_CALENDAR",
        "board_lot_status": "BLOCKED_NO_EFFECTIVE_DATED_SECURITY_SCHEDULE",
        "risk_free_rate_status": "NEEDS_RESEARCHER_APPROVAL",
        "mar_status": "NEEDS_RESEARCHER_APPROVAL",
        "risk_data_hash_status": "REQUIRED",
        "research_backtest_gate": "BLOCKED",
        "phase_f": "DO_NOT_BEGIN",
        "decision_statuses": decision_statuses,
        "decision_status_checks": decision_status_checks,
        "historical_run_allowed": False,
        "external_requests_made": False,
        "model_fit_or_inference_performed": False,
    }
    if not execution_hash_matches:
        readiness["execution_status"] = "BLOCKED_HASH_MISMATCH"
    if not model_metadata_match:
        readiness["model_provenance_status"] = "BLOCKED_MODEL_METADATA_DISAGREES_WITH_ARTIFACT"
    else:
        readiness["model_provenance_status"] = "ARTIFACT_FEATURES_MATCH_APPROVED_LIST"
    return {
        "Paths": paths,
        "Audit": inventory,
        "Readiness": readiness,
        "Execution_Data": execution,
        "Feature_Candidate": master_candidate,
        "Membership_Periods": membership_periods,
        "Membership_Changes": membership_changes,
        "Membership_2026": membership_2026,
        "Exclusions": exclusions,
        "Corporate_Actions": corporate_actions,
        "Decision_Register": decision_register,
        "Model": model,
        "Model_Metadata": metadata,
        "Execution_Sidecar": execution_sidecar,
    }


In [3]:
SOURCE_CONTEXT = inspect_local_sources()
display(SOURCE_CONTEXT['Audit'])
display(pd.Series(SOURCE_CONTEXT['Readiness'], name='ค่าที่ตรวจพบ'))
assert SOURCE_CONTEXT['Readiness']['research_backtest_gate'] == 'BLOCKED'
assert SOURCE_CONTEXT['Readiness']['phase_f'] == 'DO_NOT_BEGIN'


,Artifact,Full_Path,File_Size_Bytes,Modified_Local,SHA256,Status,Details
0,Feature_Candidate,D:\Project_Github\set_fear_index\dataset\verif...,15595544,2026-11-14T15:29:38,7424f01e92d4f507b2a5956898e1ef30050de7a50c7c4e...,HASHED_READ_ONLY,
1,Execution_Snapshot,D:\Project_Github\set_fear_index\dataset\verif...,324428,2026-11-15T11:37:12,14ca82c9a481118910952cfaaaeea08daecd6548d9d99e...,HASHED_READ_ONLY,
2,Model,D:\Project_Github\set_fear_index\models\Origin...,1969,2026-11-15T02:34:36,8ec9433bd5fa6419b7f6870f1058917b7fb3f6f0cb66cb...,HASHED_READ_ONLY,
3,Model_Metadata,D:\Project_Github\set_fear_index\models\Origin...,986,2026-11-15T02:34:36,d26da9d09b94ab50dbb803bf33d2670b93aff2c8dbaf14...,HASHED_READ_ONLY,
4,Membership_Periods,D:\Project_Github\set_fear_index\dataset\SET50...,5908,2026-11-13T03:19:32,17002a2e63259c8144589df02bcfa9c82be1ba6285e5e1...,HASHED_READ_ONLY,
5,Membership_Changes,D:\Project_Github\set_fear_index\dataset\SET50...,8020,2026-11-11T13:47:48,6d2d9b408d2c2a94e53ef5fdb7f99ce66137dddc17d0a1...,HASHED_READ_ONLY,
6,Membership_2026_Evidence,D:\Project_Github\set_fear_index\backtest_resu...,35130,2026-11-16T05:44:04,0458ce66c9628e6f711e58fcd94e65a9952cf3b618354b...,HASHED_READ_ONLY,
7,Exclusions,D:\Project_Github\set_fear_index\backtest_resu...,860,2026-11-16T03:45:14,2d10f6ba17cf712d32fc92c05f8dea7c766bb292b2f66b...,HASHED_READ_ONLY,
8,Corporate_Actions,D:\Project_Github\set_fear_index\backtest_resu...,30266,2026-11-15T13:57:18,613848ab5bd075654995b42515dc140a943eccc3e1fac9...,HASHED_READ_ONLY,
9,Decision_Register,D:\Project_Github\set_fear_index\backtest_resu...,19563,2026-11-17T04:48:04,7cc61072a253a358c7613f2e9fb29ed57a52a5c80eaf03...,HASHED_READ_ONLY,


feature_source_status                                             UNKNOWN_INSUFFICIENT_EVIDENCE
feature_candidate_status                                   CANDIDATE_NOT_ATTRIBUTED_TO_TRAINING
feature_candidate_missing_approved_columns                                                   []
model_artifact_feature_names                  [daily_return, momentum_20d, vol_z, VolRank, C...
model_metadata_features                       [daily_return, momentum_20d, vol_z, VolRank, C...
model_metadata_matches_artifact                                                            True
model_artifact_classes                                                                [0, 1, 2]
model_inference_allowed                                                                   False
execution_hash_matches_expected                                                            True
execution_source_is_vendor                                                                 True
execution_is_official_set_prints        

## ส่วนที่ 02 — จักรวาลหุ้นแบบเปลี่ยนแปลงตามเวลา

ใช้หลักฐานสมาชิก SET50 ที่ยืนยันได้เฉพาะ 2026 H1/H2 และไม่แปลงป้าย period ในไฟล์ประวัติให้เป็น effective date เอง หุ้นที่จะซื้อได้ต้องเป็นสมาชิกทั้งวันทำนาย และวันส่งคำสั่ง พร้อมผ่าน identity, corporate-action, exclusion และราคา checks การเปลี่ยนสมาชิกระหว่างสองวันจะบล็อกการซื้อ; การบังคับขายทำได้เมื่อระบุหลักทรัพย์ และสถานะ corporate action ได้แน่ชัดเท่านั้น

In [4]:
def membership_snapshot_as_of(
    membership_evidence: pd.DataFrame, target_date: Any
) -> dict[str, Any]:
    required = {
        "Period",
        "Effective_Date",
        "Symbol",
        "Local_Official_Match",
        "Status",
    }
    if not required.issubset(membership_evidence.columns):
        return {"status": "UNKNOWN", "members": set(), "reason": "SCHEMA_INVALID"}
    target = pd.Timestamp(target_date).normalize()
    evidence = membership_evidence.copy()
    evidence["Effective_Date"] = pd.to_datetime(evidence["Effective_Date"], errors="coerce")
    evidence["Local_Official_Match"] = (
        evidence["Local_Official_Match"].astype(str).str.lower().eq("true")
    )
    evidence = evidence[
        evidence["Status"].eq("VERIFIED_FROM_OFFICIAL_SET_PDF")
        & evidence["Local_Official_Match"]
    ]
    snapshots: list[dict[str, Any]] = []
    for period, group in evidence.groupby("Period", sort=True):
        effective_dates = group["Effective_Date"].dropna().dt.normalize().unique()
        if len(effective_dates) != 1:
            continue
        effective = pd.Timestamp(effective_dates[0])
        parts = str(period).split("_")
        if len(parts) != 2 or parts[1] not in {"H1", "H2"}:
            continue
        year = int(parts[0])
        expected_start = pd.Timestamp(year=year, month=1 if parts[1] == "H1" else 7, day=1)
        expected_effective = pd.Timestamp(year=year, month=1 if parts[1] == "H1" else 7, day=1)
        period_end = pd.Timestamp(year=year, month=6 if parts[1] == "H1" else 12, day=30 if parts[1] == "H1" else 31)
        symbols = set(group["Symbol"].astype(str).str.upper())
        if effective != expected_effective or len(symbols) != 50:
            continue
        if target >= expected_start and target <= period_end:
            snapshots.append(
                {"effective": effective, "members": symbols, "period": str(period)}
            )
    if not snapshots:
        return {
            "status": "UNKNOWN",
            "members": set(),
            "reason": "NO_VERIFIED_SNAPSHOT_FOR_DATE",
        }
    selected = max(snapshots, key=lambda row: row["effective"])
    return {
        "status": "VERIFIED",
        "members": selected["members"],
        "period": selected["period"],
        "effective_date": selected["effective"],
        "reason": "",
    }


def _is_excluded(
    symbol: str, decision_date: pd.Timestamp, exclusions: pd.DataFrame
) -> bool:
    if exclusions.empty:
        return False
    required = {
        "Symbol",
        "Exclusion_Start_Date",
        "Exclusion_End_Date",
        "Decision_Status",
    }
    if not required.issubset(exclusions.columns):
        raise PolicyBlockedError("EXCLUSION_SCHEMA_INVALID")
    rows = exclusions[
        exclusions["Symbol"].astype(str).str.upper().eq(symbol.upper())
        & exclusions["Decision_Status"].eq("APPROVED_RESEARCH_EXCLUSION")
    ]
    for row in rows.itertuples(index=False):
        start = pd.Timestamp(getattr(row, "Exclusion_Start_Date")).normalize()
        end = pd.Timestamp(getattr(row, "Exclusion_End_Date")).normalize()
        if start <= decision_date <= end:
            return True
    return False


def build_dynamic_universe(
    predictions: pd.DataFrame,
    predictor_date: Any,
    execution_date: Any,
    membership_evidence: pd.DataFrame,
    exclusions: pd.DataFrame,
    identity_status: Mapping[str, str],
    corporate_action_status: Mapping[str, str],
    price_status: Mapping[str, str],
    holdings: Mapping[str, float] | None = None,
    lineage: Mapping[str, str | None] | None = None,
) -> pd.DataFrame:
    predictor_day = pd.Timestamp(predictor_date).normalize()
    execution_day = pd.Timestamp(execution_date).normalize()
    if execution_day <= predictor_day:
        raise NoLookaheadError("INVALID_DATE_ORDER", "Execution must follow prediction.")
    predictor_membership = membership_snapshot_as_of(membership_evidence, predictor_day)
    execution_membership = membership_snapshot_as_of(membership_evidence, execution_day)
    held = {str(symbol).upper() for symbol, quantity in (holdings or {}).items() if quantity > 0}
    prediction_records = predictions.to_dict("records")
    supplied_symbols = {str(record["Symbol"]).upper() for record in prediction_records}
    prediction_records.extend(
        {"Symbol": symbol, "Prediction": None}
        for symbol in sorted(held.difference(supplied_symbols))
    )
    rows: list[dict[str, Any]] = []
    for record in prediction_records:
        symbol = str(record["Symbol"]).upper()
        is_holding = symbol in held
        row = {
            **record,
            "Universe_Snapshot_ID": (lineage or {}).get("Universe_Snapshot_ID"),
            "Execution_Eligibility_Snapshot_ID": (lineage or {}).get(
                "Execution_Eligibility_Snapshot_ID"
            ),
            "Predictor_Date": predictor_day,
            "Execution_Date": execution_day,
            "Predictor_Membership_Status": predictor_membership["status"],
            "Execution_Membership_Status": execution_membership["status"],
            "Predictor_Member": None,
            "Execution_Member": None,
            "Identity_Status": identity_status.get(symbol, "UNKNOWN"),
            "Corporate_Action_Status": corporate_action_status.get(symbol, "UNKNOWN"),
            "Price_Eligibility_Status": price_status.get(symbol, "UNKNOWN"),
            "Research_Excluded": _is_excluded(symbol, execution_day, exclusions),
            "Forced_Exit": False,
            "Execution_Eligibility": "BLOCKED",
            "Eligibility_Reason": "",
        }
        if predictor_membership["status"] != "VERIFIED" or execution_membership["status"] != "VERIFIED":
            row["Eligibility_Reason"] = "MEMBERSHIP_UNKNOWN_FOR_PREDICTION_OR_EXECUTION"
            rows.append(row)
            continue
        predictor_member = symbol in predictor_membership["members"]
        execution_member = symbol in execution_membership["members"]
        row["Predictor_Member"] = predictor_member
        row["Execution_Member"] = execution_member
        if is_holding and not execution_member:
            if row["Identity_Status"] != "VERIFIED":
                row["Eligibility_Reason"] = "REMOVAL_CANNOT_BE_LINKED_TO_VERIFIED_SECURITY_ID"
            elif row["Corporate_Action_Status"] not in {"VERIFIED", "VERIFIED_NO_ACTION"}:
                row["Eligibility_Reason"] = "REMOVAL_WITH_UNRESOLVED_CORPORATE_ACTION"
            else:
                row["Forced_Exit"] = True
                row["Eligibility_Reason"] = "VERIFIED_MEMBERSHIP_REMOVAL"
                row["Execution_Eligibility"] = "FORCED_EXIT_ONLY"
            rows.append(row)
            continue
        if not predictor_member or not execution_member:
            row["Eligibility_Reason"] = "NOT_MEMBER_AT_BOTH_DECISION_DATES"
        elif row["Research_Excluded"]:
            row["Eligibility_Reason"] = "APPROVED_RESEARCH_EXCLUSION"
        elif row["Identity_Status"] != "VERIFIED":
            row["Eligibility_Reason"] = "SECURITY_IDENTITY_NOT_VERIFIED"
        elif row["Corporate_Action_Status"] not in {"VERIFIED", "VERIFIED_NO_ACTION"}:
            row["Eligibility_Reason"] = "CORPORATE_ACTION_NOT_RECONCILED"
        elif row["Price_Eligibility_Status"] != "VERIFIED":
            row["Eligibility_Reason"] = "EXECUTION_PRICE_NOT_VERIFIED"
        else:
            row["Execution_Eligibility"] = "ELIGIBLE_POLICY"
            row["Eligibility_Reason"] = "ALL_POLICY_CHECKS_VERIFIED"
        rows.append(row)
    columns = [
        "Symbol",
        "Universe_Snapshot_ID",
        "Execution_Eligibility_Snapshot_ID",
        "Predictor_Date",
        "Execution_Date",
        "Predictor_Membership_Status",
        "Execution_Membership_Status",
        "Predictor_Member",
        "Execution_Member",
        "Identity_Status",
        "Corporate_Action_Status",
        "Price_Eligibility_Status",
        "Research_Excluded",
        "Forced_Exit",
        "Execution_Eligibility",
        "Eligibility_Reason",
    ]
    result = pd.DataFrame(rows)
    if result.empty:
        return pd.DataFrame(columns=columns)
    return result



## ส่วนที่ 03 — การสร้าง prediction โดยป้องกันข้อมูลอนาคต

ฟังก์ชันรับเฉพาะ feature ที่อนุมัติ เรียงคอลัมน์คงที่ และตรวจว่า Target_Date อยู่หลัง Feature_Date ตัวตนของ Feature Snapshot และความสอดคล้องระหว่างโมเดลกับ metadata ต้องผ่านก่อนจึงเรียก inference; สถานะปัจจุบันยังไม่ผ่านเงื่อนไขดังกล่าว

In [5]:
def generate_predictions(
    model: Any,
    features: pd.DataFrame,
    predictor_date: Any,
    target_date: Any,
    feature_source_status: str,
    model_metadata_match: bool,
    lineage: Mapping[str, str],
) -> pd.DataFrame:
    predictor_day = pd.Timestamp(predictor_date).normalize()
    target_day = pd.Timestamp(target_date).normalize()
    if target_day <= predictor_day:
        raise NoLookaheadError("INVALID_TARGET_DATE", "Target date must follow feature date.")
    if feature_source_status != "VERIFIED":
        raise PolicyBlockedError("FEATURE_SOURCE_UNVERIFIED", feature_source_status)
    if not model_metadata_match:
        raise PolicyBlockedError("MODEL_PROVENANCE_INCONSISTENT")
    missing = [column for column in FEATURES if column not in features.columns]
    if missing:
        raise PolicyBlockedError("FEATURE_SCHEMA_INVALID", str(missing))
    for field_name in ("Model_Version", "Prediction_Snapshot_ID"):
        if not lineage.get(field_name):
            raise PolicyBlockedError("PROVENANCE_INCOMPLETE", field_name)
    if "Date" in features:
        dates = pd.to_datetime(features["Date"]).dt.normalize()
        if not dates.eq(predictor_day).all():
            raise PolicyBlockedError("FEATURE_DATE_MISMATCH")
    ordered = features.loc[:, FEATURES].copy()
    probabilities = np.asarray(model.predict_proba(ordered), dtype=float)
    classes = [int(value) for value in model.classes_]
    if probabilities.shape != (len(features), len(classes)):
        raise PolicyBlockedError("MODEL_OUTPUT_SCHEMA_INVALID")
    if not np.isfinite(probabilities).all():
        raise PolicyBlockedError("MODEL_OUTPUT_NONFINITE")
    results: list[dict[str, Any]] = []
    for index, (_, source) in enumerate(features.iterrows()):
        record = {
            "Feature_Date": predictor_day,
            "Target_Date": target_day,
            "Symbol": str(source["Symbol"]).upper() if "Symbol" in source else "",
            "Prediction": classes[int(np.argmax(probabilities[index]))],
            "Model_Version": lineage["Model_Version"],
            "Prediction_Snapshot_ID": lineage["Prediction_Snapshot_ID"],
            "Model_Artifact_SHA256": lineage.get("Model_Artifact_SHA256"),
            "vol_z": source.get("vol_z", np.nan),
            "Universe_Snapshot_ID": lineage.get("Universe_Snapshot_ID"),
            "Execution_Eligibility_Snapshot_ID": lineage.get(
                "Execution_Eligibility_Snapshot_ID"
            ),
            "Allocation_Policy_Version": lineage.get("Allocation_Policy_Version"),
        }
        for class_index, class_value in enumerate(classes):
            record[f"Probability_{class_value}"] = float(probabilities[index, class_index])
        results.append(record)
    return pd.DataFrame(results)


## ส่วนที่ 04 — นโยบายสัญญาณทั้งเก้าแบบ

ใช้ 9 กลยุทธ์จาก Cartesian Product ของคำสั่ง Panic × FOMO โดย Normal เป็น Hold เสมอ: S1 Buy/Buy, S2 Buy/Sell, S3 Buy/Hold, S4 Sell/Buy, S5 Sell/Sell, S6 Sell/Hold, S7 Hold/Buy, S8 Hold/Sell และ S9 Hold/Hold กลยุทธ์แยกจาก Allocation และไม่มี threshold หรือ Top-K เพิ่มเติม

In [6]:
def strategy_signals(
    predictions: pd.DataFrame,
    current_weights: Mapping[str, float],
    strategy_id: str,
    parameters: Mapping[str, Any],
) -> pd.DataFrame:
    if strategy_id not in STRATEGIES:
        raise ValueError(f"Unknown strategy: {strategy_id}")
    required = ("entry_class", "exit_class", "entry_threshold", "exit_threshold", "top_k")
    missing = [key for key in required if key not in parameters]
    if missing:
        raise PolicyBlockedError("STRATEGY_PARAMETERS_INCOMPLETE", str(missing))
    entry_class = int(parameters["entry_class"])
    exit_class = int(parameters["exit_class"])
    entry_threshold = float(parameters["entry_threshold"])
    exit_threshold = float(parameters["exit_threshold"])
    top_k = int(parameters["top_k"])
    if not (0 <= entry_threshold <= 1 and 0 <= exit_threshold <= 1 and top_k > 0):
        raise ValueError("Thresholds must be in [0, 1] and top_k must be positive.")
    entry_probability = f"Probability_{entry_class}"
    exit_probability = f"Probability_{exit_class}"
    if entry_probability not in predictions or exit_probability not in predictions:
        raise PolicyBlockedError("PREDICTION_CLASS_PROBABILITY_MISSING")
    frame = predictions.copy()
    frame["Symbol"] = frame["Symbol"].astype(str).str.upper()
    frame["Current_Weight"] = frame["Symbol"].map(current_weights).fillna(0.0).astype(float)
    frame["Entry_Score"] = pd.to_numeric(frame[entry_probability], errors="coerce")
    frame["Exit_Score"] = pd.to_numeric(frame[exit_probability], errors="coerce")
    if not np.isfinite(frame[["Entry_Score", "Exit_Score"]].to_numpy(dtype=float)).all():
        raise PolicyBlockedError("PREDICTION_PROBABILITY_NONFINITE")
    frame["Action"] = "HOLD"
    frame["Target_Weight"] = frame["Current_Weight"]
    frame["Signal_Reason"] = "NO_CHANGE"
    entry = frame["Prediction"].astype(int).eq(entry_class) & frame["Entry_Score"].ge(entry_threshold)
    exit_signal = frame["Prediction"].astype(int).eq(exit_class) & frame["Exit_Score"].ge(exit_threshold)
    rank_columns = ["Entry_Score", "Symbol"]
    rank_ascending = [False, True]
    if strategy_id == "S3_TopK_TieBreak" and "vol_z" in frame.columns:
        rank_columns = ["Entry_Score", "vol_z", "Symbol"]
        rank_ascending = [False, True, True]
    ranked = frame.loc[entry].sort_values(
        rank_columns, ascending=rank_ascending, kind="stable"
    ).index.tolist()
    selected = set(ranked[:top_k])
    default_target = min(
        MAX_POSITION_WEIGHT,
        float(parameters.get("target_weight", 1.0 / max(top_k, 1))),
    )
    for index, row in frame.iterrows():
        held = row["Current_Weight"] > 0
        is_entry = bool(entry.loc[index])
        is_exit = bool(exit_signal.loc[index])
        chosen = index in selected
        action = "HOLD"
        reason = "NO_CHANGE"
        target = float(row["Current_Weight"])
        if strategy_id == "S1_Threshold":
            if is_entry:
                action, reason, target = "BUY", "ENTRY_THRESHOLD", default_target
            elif is_exit:
                action, reason, target = "SELL", "EXIT_THRESHOLD", 0.0
        elif strategy_id in {"S2_TopK", "S3_TopK_TieBreak"}:
            if chosen:
                action, reason, target = "BUY", "TOP_K_ENTRY", default_target
            elif held and is_exit:
                action, reason, target = "SELL", "EXIT_THRESHOLD", 0.0
            elif held and not chosen:
                action, reason, target = "SELL", "OUTSIDE_TOP_K", 0.0
        elif strategy_id == "S4_Hysteresis":
            if held and row["Entry_Score"] < exit_threshold:
                action, reason, target = "SELL", "HYSTERESIS_EXIT", 0.0
            elif held:
                action, reason = "HOLD", "HYSTERESIS_HOLD"
            elif row["Entry_Score"] >= entry_threshold:
                action, reason, target = "BUY", "HYSTERESIS_ENTRY", default_target
        elif strategy_id == "S5_ExitOnNeutral":
            if is_entry:
                action, reason, target = "BUY", "ENTRY_THRESHOLD", default_target
            elif held:
                action, reason, target = (
                    "SELL",
                    "EXIT_CLASS" if is_exit else "EXIT_ON_NEUTRAL",
                    0.0,
                )
        elif strategy_id == "S6_EntryRankExitThreshold":
            if chosen:
                action, reason, target = "BUY", "ENTRY_TOP_K", default_target
            elif held and is_exit:
                action, reason, target = "SELL", "EXIT_THRESHOLD", 0.0
        elif strategy_id == "S7_EntryThresholdExitRank":
            if is_entry:
                action, reason, target = "BUY", "ENTRY_THRESHOLD", default_target
            elif held and not chosen:
                action, reason, target = "SELL", "EXIT_RANK"
        elif strategy_id == "S8_BandRebalance":
            if is_entry:
                desired = float(parameters.get("band_target_weight", 1.0 / top_k))
                band = float(parameters.get("rebalance_band", 0.05))
                current = float(row["Current_Weight"])
                target = desired
                if desired - current > band:
                    action, reason = "BUY", "BELOW_TARGET_WEIGHT_BAND"
                elif current - desired > band and held:
                    action, reason, target = "SELL", "ABOVE_TARGET_WEIGHT_BAND", 0.0
            elif held and is_exit:
                action, reason, target = "SELL", "EXIT_THRESHOLD", 0.0
        elif strategy_id == "S9_Staged":
            stage_one = float(parameters.get("stage_one_threshold", entry_threshold))
            stage_two = float(parameters.get("stage_two_threshold", min(1.0, stage_one + 0.1)))
            stage_one_weight = float(parameters.get("stage_one_weight", 0.5 / top_k))
            stage_two_weight = float(parameters.get("stage_two_weight", 1.0 / top_k))
            if is_exit:
                action, reason, target = "SELL", "STAGED_EXIT", 0.0
            elif row["Entry_Score"] >= stage_two and float(row["Current_Weight"]) < stage_two_weight:
                action, reason, target = "BUY", "STAGE_TWO", stage_two_weight
            elif row["Entry_Score"] >= stage_one and float(row["Current_Weight"]) < stage_one_weight:
                action, reason, target = "BUY", "STAGE_ONE", stage_one_weight
        frame.at[index, "Action"] = action
        frame.at[index, "Target_Weight"] = target
        frame.at[index, "Signal_Reason"] = reason
    frame["Strategy_ID"] = strategy_id
    frame["Target_Weight"] = frame["Target_Weight"].clip(lower=0.0, upper=MAX_POSITION_WEIGHT)
    return frame


## ส่วนที่ 05 — ตัวจัดสรรเงินเจ็ดวิธี

รองรับ EW, IV, ERC, HRP, MinVar, CVaR และ MPT โดยใช้ common-complete ordered risk panel ก่อน cutoff เดียวกัน ห้ามเติมค่า ลบหลักทรัพย์ หรือใช้ pairwise ข้อมูล IV/ERC/HRP/MinVar/MPT ต้องมีอย่างน้อย 60 ข้อมูล และ CVaR 252 ข้อมูล ทุกวิธีจำกัดน้ำหนักรายตัวไม่เกิน 25% ของเงินสดที่นำมาจัดสรร เงินสดส่วนที่จัดสรรไม่ได้หรือเหลือจากการซื้อจะคงเป็นเงินสด ไม่มีข้อกำหนดสำรองเงินสด 50%

`Risk_Data_Hash` ระบุ common panel ร่วม ส่วน `Method_Input_Hash` ระบุช่วง 60/252 รายการจริงที่วิธีนั้นใช้ การแยก hash นี้ทำให้ lineage ของ panel และหน้าต่างคำนวณตรวจสอบได้

In [7]:
@dataclass
class AllocationResult:
    method: str
    status: str
    reason: str
    weights: dict[str, float] = field(default_factory=dict)
    raw_weights: dict[str, float] = field(default_factory=dict)
    risky_sleeve_weights: dict[str, float] = field(default_factory=dict)
    allocated_amounts: dict[str, float] = field(default_factory=dict)
    reserve_cash: float = 0.0
    deployable_cash: float = 0.0
    risky_budget: float = 0.0
    risk_data_hash: str | None = None
    method_input_hash: str | None = None
    lookback_observations: int = 0
    risk_cutoff: str | None = None
    solver_status: str | None = None
    solver_diagnostics: dict[str, Any] = field(default_factory=dict)
    max_position_weight: float = MAX_POSITION_WEIGHT


def build_common_risk_panel(
    returns: pd.DataFrame,
    symbols: Sequence[str],
    decision_date: Any,
    verified_sessions: Sequence[Any],
    calendar_verified: bool,
    calendar_id: str | None,
) -> tuple[pd.DataFrame, str, int]:
    if not calendar_verified or not calendar_id:
        raise PolicyBlockedError("CALENDAR_NOT_VERIFIED")
    if not symbols:
        raise PolicyBlockedError("EMPTY_RISK_UNIVERSE")
    required = {"Date", "Symbol", "Return"}
    if not required.issubset(returns.columns):
        raise PolicyBlockedError("RETURN_SCHEMA_INVALID")
    decision_day = pd.Timestamp(decision_date).normalize()
    session_days = pd.DatetimeIndex(pd.to_datetime(list(verified_sessions))).normalize().unique().sort_values()
    candidates = session_days[session_days < decision_day]
    source = returns.loc[:, ["Date", "Symbol", "Return"]].copy()
    source["Date"] = pd.to_datetime(source["Date"]).dt.normalize()
    source["Symbol"] = source["Symbol"].astype(str).str.upper()
    symbol_order = [str(symbol).upper() for symbol in symbols]
    source = source[
        source["Date"].isin(candidates) & source["Symbol"].isin(symbol_order)
    ]
    require_unique_key(source, ["Date", "Symbol"], "Risk return panel")
    source["Return"] = pd.to_numeric(source["Return"], errors="coerce")
    wide = source.pivot(index="Date", columns="Symbol", values="Return")
    wide = wide.reindex(index=candidates, columns=symbol_order)
    values = wide.to_numpy(dtype=float)
    complete_mask = np.isfinite(values).all(axis=1)
    common = wide.loc[complete_mask].copy()
    dropped_dates = int(len(candidates) - len(common))
    if common.empty:
        raise PolicyBlockedError("COMMON_COMPLETE_SAMPLE_EMPTY")
    common.index.name = "Date"
    return common, canonical_frame_hash(common), dropped_dates


def _validate_covariance(matrix: np.ndarray) -> tuple[np.ndarray, float]:
    covariance = np.cov(matrix, rowvar=False, ddof=1)
    covariance = np.atleast_2d(covariance).astype(float)
    if not np.isfinite(covariance).all():
        raise PolicyBlockedError("COVARIANCE_NONFINITE")
    if not np.allclose(covariance, covariance.T, rtol=1e-10, atol=1e-12):
        raise PolicyBlockedError("COVARIANCE_NOT_SYMMETRIC")
    eigenvalues = np.linalg.eigvalsh(covariance)
    scale = max(float(np.max(np.abs(eigenvalues))), 1.0)
    if float(np.min(eigenvalues)) <= 1e-12 * scale:
        raise PolicyBlockedError("COVARIANCE_NOT_POSITIVE_DEFINITE")
    condition = float(np.linalg.cond(covariance))
    if not np.isfinite(condition) or condition > ERC_MAX_CONDITION_NUMBER:
        raise PolicyBlockedError("COVARIANCE_ILL_CONDITIONED", str(condition))
    return covariance, condition


def _erc_relative_contributions(weights: np.ndarray, covariance: np.ndarray) -> np.ndarray:
    variance = float(weights @ covariance @ weights)
    if not np.isfinite(variance) or variance <= 0:
        raise PolicyBlockedError("ERC_PORTFOLIO_VARIANCE_INVALID")
    contributions = weights * (covariance @ weights)
    total = float(contributions.sum())
    if total <= 0 or not np.isfinite(total):
        raise PolicyBlockedError("ERC_RISK_CONTRIBUTION_INVALID")
    return contributions / total


def fractional_tail_cvar(
    losses: Sequence[float], session_dates: Sequence[Any], alpha: float = 0.95
) -> dict[str, Any]:
    values = np.asarray(losses, dtype=float)
    dates = pd.to_datetime(list(session_dates))
    if len(values) != len(dates) or len(values) == 0 or not np.isfinite(values).all():
        raise PolicyBlockedError("CVAR_SCENARIOS_INVALID")
    if not 0.0 < alpha < 1.0:
        raise ValueError("CVaR alpha must be between zero and one.")
    tail_mass = (1.0 - alpha) * len(values)
    if tail_mass <= 0 or tail_mass > len(values):
        raise PolicyBlockedError("CVAR_TAIL_MASS_INVALID")
    order = sorted(range(len(values)), key=lambda index: (-values[index], dates[index]))
    boundary_index = int(math.floor(tail_mass))
    fractional_mass = tail_mass - boundary_index
    strictly_worse = order[:boundary_index]
    boundary_loss = values[order[boundary_index]] if fractional_mass > 0 else values[order[boundary_index - 1]]
    worse_value = float(values[strictly_worse].sum()) if strictly_worse else 0.0
    boundary_group = [index for index in order if values[index] == boundary_loss]
    remaining_mass = tail_mass - len(strictly_worse)
    per_tied_weight = remaining_mass / len(boundary_group)
    total = worse_value + float(values[boundary_group].sum()) * per_tied_weight
    return {
        "CVaR_Objective": total / tail_mass,
        "Tail_Mass": tail_mass,
        "Tail_Cutoff_Loss": float(boundary_loss),
        "Tie_Group_Size": len(boundary_group),
        "Fractional_Weight_Per_Tied_Observation": per_tied_weight,
        "Tail_Scenario_Order": [dates[index].date().isoformat() for index in order],
    }


def _capped_sleeve_weights(
    raw: np.ndarray, total_cash: float, preserve_scale: bool
) -> tuple[np.ndarray, float, float]:
    if not np.isfinite(raw).all() or np.any(raw < 0) or raw.sum() <= 0:
        raise PolicyBlockedError("RAW_WEIGHTS_INVALID")
    if not np.isfinite(total_cash) or total_cash < 0:
        raise ValueError("Cash must be finite and nonnegative.")
    count = len(raw)
    base = raw.copy()
    if not preserve_scale:
        base /= base.sum()
    target_weight = min(float(base.sum()), 1.0, MAX_POSITION_WEIGHT * len(base))
    per_asset_cap = MAX_POSITION_WEIGHT
    allocations = np.zeros(count, dtype=float)
    active = np.ones(count, dtype=bool)
    remaining = target_weight
    for _ in range(count + 1):
        if remaining <= 1e-12 or not active.any():
            break
        proportions = base[active] / base[active].sum()
        proposed = proportions * remaining
        active_indices = np.flatnonzero(active)
        capped = proposed > (per_asset_cap - allocations[active_indices])
        if not capped.any():
            allocations[active_indices] += proposed
            remaining = 0.0
            break
        for local_index in np.flatnonzero(capped):
            index = active_indices[local_index]
            addition = max(0.0, per_asset_cap - allocations[index])
            allocations[index] += addition
            remaining -= addition
            active[index] = False
    allocated = float(allocations.sum() * total_cash)
    weights = allocations
    return weights, max(0.0, total_cash - allocated), allocated


def _allocation_result(
    method: str,
    symbols: Sequence[str],
    raw_weights: np.ndarray,
    total_cash: float,
    risk_data_hash: str | None = None,
    method_input_hash: str | None = None,
    lookback: int = 0,
    cutoff: str | None = None,
    solver_status: str | None = None,
    preserve_scale: bool = False,
    risky_sleeve_weights: np.ndarray | None = None,
    solver_diagnostics: Mapping[str, Any] | None = None,
) -> AllocationResult:
    normalized = np.asarray(raw_weights, dtype=float)
    normalized = np.maximum(normalized, 0.0)
    if normalized.sum() <= 0:
        return AllocationResult(
            method=method,
            status="UNAVAILABLE",
            reason="NO_POSITIVE_WEIGHTS",
            reserve_cash=float(total_cash),
            risk_data_hash=risk_data_hash,
            method_input_hash=method_input_hash,
            lookback_observations=lookback,
            risk_cutoff=cutoff,
            solver_status=solver_status,
        )
    if not preserve_scale:
        normalized /= normalized.sum()
    weights, reserve_cash, allocated = _capped_sleeve_weights(
        normalized, total_cash, preserve_scale
    )
    raw_weight_map = {
        str(symbol).upper(): float(weight)
        for symbol, weight in zip(symbols, normalized)
    }
    result_weights = {
        str(symbol).upper(): float(weight)
        for symbol, weight in zip(symbols, weights)
        if weight > 0
    }
    amounts = {
        str(symbol).upper():                 float(weight * total_cash)
        for symbol, weight in zip(symbols, weights)
        if weight > 0
    }
    return AllocationResult(
        method=method,
        status=        "ALLOCATED" if allocated > 0 else "NO_DEPLOYABLE_CASH",
        reason="E3_METHOD_INPUTS_VALID" if allocated > 0 else "NO_DEPLOYABLE_CASH",
        weights=result_weights,
        raw_weights=raw_weight_map,
        risky_sleeve_weights=(
            {
                str(symbol).upper(): float(weight)
                for symbol, weight in zip(symbols, risky_sleeve_weights)
            }
            if risky_sleeve_weights is not None
            else {}
        ),
        allocated_amounts=amounts,
        reserve_cash=reserve_cash,
        deployable_cash=float(total_cash),
        risky_budget=allocated,
        risk_data_hash=risk_data_hash,
        method_input_hash=method_input_hash,
        lookback_observations=lookback,
        risk_cutoff=cutoff,
        solver_status=solver_status,
        solver_diagnostics=dict(solver_diagnostics or {}),
    )


def allocate_ew(symbols: Sequence[str], total_cash: float) -> AllocationResult:
    if not symbols:
        return AllocationResult("EW", "UNAVAILABLE", "EMPTY_UNIVERSE", reserve_cash=total_cash)
    return _allocation_result(
        "EW",
        symbols,
        np.ones(len(symbols)),
        total_cash,
    )


def allocate_method(
    method: str,
    symbols: Sequence[str],
    total_cash: float,
    returns: pd.DataFrame,
    decision_date: Any,
    verified_sessions: Sequence[Any],
    calendar_verified: bool,
    calendar_id: str | None,
    alpha: float = 0.95,
    risk_aversion: float = 5.0,
) -> AllocationResult:
    if method not in {"IV", "ERC", "HRP", "MinVar", "CVaR", "MPT"}:
        raise ValueError(f"Unknown risk allocation method: {method}")
    lookback = 252 if method == "CVaR" else 60
    risk_hash: str | None = None
    method_hash: str | None = None
    observed = 0
    cutoff: str | None = None
    solver_status: str | None = None
    solver_diagnostics: dict[str, Any] = {}
    try:
        common, risk_hash, _ = build_common_risk_panel(
            returns,
            symbols,
            decision_date,
            verified_sessions,
            calendar_verified,
            calendar_id,
        )
        observed = len(common)
        method_panel = common.tail(lookback)
        method_hash = canonical_frame_hash(method_panel)
        if not method_panel.empty:
            cutoff = pd.Timestamp(method_panel.index[-1]).date().isoformat()
        if len(common) < lookback:
            raise PolicyBlockedError(
                "INSUFFICIENT_COMMON_OBSERVATIONS",
                f"required={lookback}, observed={len(common)}",
            )
        matrix = method_panel.to_numpy(dtype=float)
        solver_status = "NOT_REQUIRED"
        if method == "IV":
            volatility = np.std(matrix, axis=0, ddof=1)
            if not np.isfinite(volatility).all() or np.any(volatility <= 0):
                raise PolicyBlockedError("IV_VOLATILITY_INVALID")
            raw = 1.0 / volatility
        elif method in {"ERC", "MinVar", "MPT"}:
            covariance, _ = _validate_covariance(matrix)
            count = len(symbols)
            if method == "ERC":
                initial = np.full(count, 1.0 / count)

                def objective(weights: np.ndarray) -> float:
                    contributions = _erc_relative_contributions(weights, covariance)
                    return float(np.sum((contributions - 1.0 / count) ** 2))

                optimized = minimize(
                    objective,
                    initial,
                    method="SLSQP",
                    bounds=[(0.0, 1.0)] * count,
                    constraints=[{"type": "eq", "fun": lambda weights: np.sum(weights) - 1.0}],
                    options={"ftol": 1e-8, "maxiter": 10000, "disp": False},
                )
                solver_status = str(optimized.message)
                if not optimized.success:
                    raise PolicyBlockedError("ERC_SOLVER_FAILED", solver_status)
                raw = np.maximum(optimized.x, 0.0)
                if (
                    not np.isfinite(optimized.fun)
                    or not np.isfinite(raw).all()
                    or np.any(raw < 0)
                    or abs(float(raw.sum()) - 1.0) > 1e-8
                ):
                    raise PolicyBlockedError("ERC_CONSTRAINT_CHECK_FAILED")
                raw /= raw.sum()
                solver_diagnostics = {
                    "Solver": "SLSQP",
                    "Iterations": int(optimized.nit),
                    "Objective": float(optimized.fun),
                    "Risky_Sleeve_Weight_Sum": float(raw.sum()),
                }
            elif method == "MinVar":
                target_exposure = min(1.0, 0.25 * count)
                initial = np.full(count, target_exposure / count)
                optimized = minimize(
                    lambda weights: float(weights @ covariance @ weights),
                    initial,
                    method="SLSQP",
                    bounds=[(0.0, 0.25)] * count,
                    constraints=[{"type": "eq", "fun": lambda weights: np.sum(weights) - target_exposure}],
                    options={"ftol": 1e-10, "maxiter": 10000, "disp": False},
                )
                solver_status = str(optimized.message)
                if not optimized.success:
                    raise PolicyBlockedError("MINVAR_SOLVER_FAILED", solver_status)
                raw = np.maximum(optimized.x, 0.0)
                if (
                    not np.isfinite(optimized.fun)
                    or not np.isfinite(raw).all()
                    or np.any(raw < -1e-8)
                    or np.any(raw > 0.25 + 1e-8)
                    or abs(float(raw.sum()) - target_exposure) > 1e-8
                ):
                    raise PolicyBlockedError("MINVAR_CONSTRAINT_CHECK_FAILED")
                solver_diagnostics = {
                    "Solver": "SLSQP",
                    "Iterations": int(optimized.nit),
                    "Objective": float(optimized.fun),
                    "Weight_Sum_Residual": float(abs(raw.sum() - target_exposure)),
                    "Maximum_Bound_Violation": float(max(0.0, np.max(raw - 0.25))),
                }
            else:
                if not math.isclose(risk_aversion, 5.0, rel_tol=0.0, abs_tol=1e-12):
                    raise PolicyBlockedError("MPT_RISK_AVERSION_MISMATCH")
                means = np.mean(matrix, axis=0)
                initial = np.zeros(count)
                optimized = minimize(
                    lambda weights: float(
                        -(means @ weights - risk_aversion * (weights @ covariance @ weights))
                    ),
                    initial,
                    method="SLSQP",
                    bounds=[(0.0, 0.25)] * count,
                    constraints=[{"type": "ineq", "fun": lambda weights: 1.0 - np.sum(weights)}],
                    options={"ftol": 1e-10, "maxiter": 10000, "disp": False},
                )
                solver_status = str(optimized.message)
                if not optimized.success:
                    raise PolicyBlockedError("MPT_SOLVER_FAILED", solver_status)
                raw = np.maximum(optimized.x, 0.0)
                if (
                    not np.isfinite(optimized.fun)
                    or not np.isfinite(raw).all()
                    or np.any(raw < -1e-8)
                    or np.any(raw > 0.25 + 1e-8)
                    or raw.sum() > 1.0 + 1e-8
                ):
                    raise PolicyBlockedError("MPT_CONSTRAINT_CHECK_FAILED")
                solver_diagnostics = {
                    "Solver": "SLSQP",
                    "Iterations": int(optimized.nit),
                    "Objective": float(-optimized.fun),
                    "Weight_Sum": float(raw.sum()),
                    "Weight_Sum_Residual": float(max(0.0, raw.sum() - 1.0)),
                }
                if raw.sum() <= 1e-12:
                    return AllocationResult(
                        method=method,
                        status="ALLOCATED",
                        reason="OPTIMAL_ALL_CASH",
                        reserve_cash=float(total_cash),
                        deployable_cash=float(total_cash),
                        risk_data_hash=risk_hash,
                        method_input_hash=method_hash,
                        lookback_observations=observed,
                        risk_cutoff=cutoff,
                        solver_status=solver_status,
                        solver_diagnostics=solver_diagnostics,
                    )
        elif method == "HRP":
            covariance, condition = _validate_covariance(matrix)
            count = len(symbols)
            if count == 1:
                raw = np.ones(1)
            else:
                correlation = method_panel.corr(method="pearson").to_numpy(dtype=float)
                if not np.isfinite(correlation).all():
                    raise PolicyBlockedError("HRP_CORRELATION_NONFINITE")
                if np.any(correlation < -1.0 - 1e-10) or np.any(correlation > 1.0 + 1e-10):
                    raise PolicyBlockedError("HRP_CORRELATION_OUT_OF_RANGE")
                distances = np.sqrt(np.maximum(0.0, (1.0 - correlation) / 2.0))
                np.fill_diagonal(distances, 0.0)
                condensed = squareform(distances, checks=True)
                tree = linkage(condensed, method="single", optimal_ordering=True)
                order = leaves_list(tree)
                ordered_symbols = [symbols[index] for index in order]
                ordered_cov = covariance[np.ix_(order, order)]
                ordered_weights = np.ones(count, dtype=float)

                def cluster_variance(indices: np.ndarray) -> float:
                    sub = ordered_cov[np.ix_(indices, indices)]
                    variances = np.diag(sub)
                    if not np.isfinite(variances).all() or np.any(variances <= 0):
                        raise PolicyBlockedError("HRP_CLUSTER_VARIANCE_INVALID")
                    weights = 1.0 / variances
                    weights /= weights.sum()
                    return float(weights @ sub @ weights)

                clusters = [np.arange(count)]
                while any(len(cluster) > 1 for cluster in clusters):
                    next_clusters: list[np.ndarray] = []
                    for cluster in clusters:
                        if len(cluster) <= 1:
                            next_clusters.append(cluster)
                            continue
                        split = len(cluster) // 2
                        left, right = cluster[:split], cluster[split:]
                        left_var, right_var = cluster_variance(left), cluster_variance(right)
                        if left_var + right_var <= 0:
                            raise PolicyBlockedError("HRP_CLUSTER_VARIANCE_INVALID")
                        left_fraction = 1.0 - left_var / (left_var + right_var)
                        ordered_weights[left] *= left_fraction
                        ordered_weights[right] *= 1.0 - left_fraction
                        next_clusters.extend([left, right])
                    clusters = next_clusters
                raw = np.zeros(count, dtype=float)
                for position, source_index in enumerate(order):
                    raw[source_index] = ordered_weights[position]
            solver_diagnostics = {
                "Estimator": "CenteredSampleCorrelationAndCovariance",
                "Linkage": "single",
                "Condition_Number": float(condition),
            }
        elif method == "CVaR":
            if not (0.0 < alpha < 1.0):
                raise ValueError("CVaR alpha must be between zero and one.")
            count, observation_count = matrix.shape[1], matrix.shape[0]
            target_exposure = min(1.0, 0.25 * count)
            objective = np.r_[
                np.zeros(count),
                1.0,
                np.full(observation_count, 1.0 / ((1.0 - alpha) * observation_count)),
            ]
            constraints = np.zeros((observation_count + 1, count + 1 + observation_count))
            constraints[:observation_count, :count] = -matrix
            constraints[:observation_count, count] = -1.0
            constraints[:observation_count, count + 1 :] = -np.eye(observation_count)
            constraints[-1, :count] = 1.0
            bounds = [(0.0, 0.25)] * count + [(None, None)] + [
                (0.0, None)
            ] * observation_count
            right_hand = np.r_[np.zeros(observation_count), target_exposure]
            equalities = np.zeros((1, len(objective)))
            equalities[0, :count] = 1.0
            optimized = linprog(
                objective,
                A_ub=constraints,
                b_ub=right_hand,
                A_eq=equalities,
                b_eq=np.array([target_exposure]),
                bounds=bounds,
                method="highs",
                options={
                    "maxiter": 100000,
                    "primal_feasibility_tolerance": 1e-8,
                    "dual_feasibility_tolerance": 1e-8,
                },
            )
            solver_status = str(optimized.message)
            if not optimized.success:
                raise PolicyBlockedError("CVAR_SOLVER_FAILED", solver_status)
            raw = np.maximum(optimized.x[:count], 0.0)
            if (
                not np.isfinite(optimized.fun)
                or not np.isfinite(raw).all()
                or np.any(raw < -1e-8)
                or np.any(raw > 0.25 + 1e-8)
                or abs(float(raw.sum()) - target_exposure) > 1e-8
            ):
                raise PolicyBlockedError("CVAR_CONSTRAINT_CHECK_FAILED")
            losses = -(matrix @ raw)
            tail_diagnostics = fractional_tail_cvar(losses, method_panel.index, alpha)
            solver_diagnostics = {
                "Solver": "HiGHS",
                "Iterations": int(getattr(optimized, "nit", 0)),
                **tail_diagnostics,
                "Weight_Sum": float(raw.sum()),
                "Weight_Sum_Residual": float(abs(raw.sum() - target_exposure)),
                "Maximum_Bound_Violation": float(max(0.0, np.max(raw - 0.25))),
            }
        else:
            raw = np.array([])
        result = _allocation_result(
            method,
            symbols,
            raw,
            total_cash,
            risk_hash,
            method_hash,
            lookback,
            cutoff,
            solver_status,
            preserve_scale=method in {"MinVar", "CVaR", "MPT"},
            risky_sleeve_weights=raw if method == "ERC" else None,
            solver_diagnostics=solver_diagnostics,
        )
        if method == "ERC" and result.status == "ALLOCATED":
            actual = np.array([result.weights.get(str(symbol).upper(), 0.0) for symbol in symbols])
            covariance, _ = _validate_covariance(matrix)
            relative_rc = _erc_relative_contributions(actual, covariance)
            deviation = float(
                np.max(np.abs(relative_rc - 1.0 / len(symbols)) / (1.0 / len(symbols)))
            )
            if deviation > ERC_MAX_RELATIVE_RC_DEVIATION:
                raise PolicyBlockedError("ERC_CONTRIBUTION_DEVIATION", str(deviation))
            result.solver_diagnostics["Post_Cap_Relative_RC"] = relative_rc.tolist()
            result.solver_diagnostics["Post_Cap_Relative_RC_Deviation"] = deviation
        return result
    except PolicyBlockedError as error:
        return AllocationResult(
            method=method,
            status="UNAVAILABLE",
            reason=f"{error.reason}:{error.detail}" if error.detail else error.reason,
            reserve_cash=float(total_cash),
            deployable_cash=float(total_cash),
            risk_data_hash=risk_hash,
            method_input_hash=method_hash,
            lookback_observations=observed,
            risk_cutoff=cutoff,
            solver_status=solver_status,
            solver_diagnostics=solver_diagnostics,
        )


def allocate_iv(**kwargs: Any) -> AllocationResult:
    return allocate_method("IV", **kwargs)


def allocate_erc(**kwargs: Any) -> AllocationResult:
    return allocate_method("ERC", **kwargs)


def allocate_hrp(**kwargs: Any) -> AllocationResult:
    return allocate_method("HRP", **kwargs)


def allocate_minvar(**kwargs: Any) -> AllocationResult:
    return allocate_method("MinVar", **kwargs)


def allocate_cvar(**kwargs: Any) -> AllocationResult:
    return allocate_method("CVaR", **kwargs)


def allocate_mpt(**kwargs: Any) -> AllocationResult:
    return allocate_method("MPT", **kwargs)


## ส่วนที่ 06 — การส่งคำสั่งและการบันทึกบัญชี

ประมวลผลขายก่อนซื้อ จัดสรรงบก่อนอ่านราคา Open สำหรับคำสั่งซื้อ ไม่มีการแทน Open ที่หายด้วย Close หรือราคาเดิม การกำหนดจำนวนหุ้นต้องใช้ board-lot schedule ที่ยืนยันและมีช่วงวันที่ใช้บังคับ; ถ้าไม่มีให้บล็อก บันทึก notional, fee, cash delta, no-fill reason และ lineage ใช้ Slippage 0.10% ต่อด้านตามกติกาที่อนุมัติ: BUY = Open × 1.001 และ SELL = Open × 0.999

In [8]:
TRANSACTION_COLUMNS = [
    "Strategy_ID",
    "Allocation_Method",
    "Predictor_Date",
    "Execution_Date",
    "Symbol",
    "Side",
    "Quantity",
    "Raw_Open",
    "Fill_Price",
    "Gross_Notional",
    "Fee_Rate",
    "Fee",
    "Slippage_Cost",
    "Slippage_Status",
    "Cash_Delta",
    "Cash_Debit_or_Credit",
    "Cash_After",
    "Status",
    "Reason",
    "Actual_Executed_Weight",
    "Allocation_Reason",
    "Risk_Data_Hash",
    "Method_Input_Hash",
    "Universe_Snapshot_ID",
    "Prediction_Snapshot_ID",
    "Execution_Eligibility_Snapshot_ID",
    "Model_Version",
    "Allocation_Policy_Version",
]


def _lot_rule_for(
    symbol: str,
    decision_date: Any,
    board_lot_schedule: Mapping[str, Mapping[str, Any]],
) -> dict[str, Any]:
    rule = dict(board_lot_schedule.get(symbol.upper(), {}))
    day = pd.Timestamp(decision_date).normalize()
    if rule.get("Status") != "VERIFIED":
        raise PolicyBlockedError("BOARD_LOT_SCHEDULE_UNVERIFIED", symbol)
    if "Effective_From" not in rule or "Effective_To" not in rule:
        raise PolicyBlockedError("BOARD_LOT_EFFECTIVE_DATES_MISSING", symbol)
    if not (
        pd.Timestamp(rule["Effective_From"]).normalize()
        <= day
        <= pd.Timestamp(rule["Effective_To"]).normalize()
    ):
        raise PolicyBlockedError("BOARD_LOT_RULE_OUT_OF_DATE", symbol)
    multiple = int(rule.get("Quantity_Multiple", 0))
    minimum = int(rule.get("Minimum_Order_Quantity", 0))
    if multiple <= 0 or minimum <= 0:
        raise PolicyBlockedError("BOARD_LOT_RULE_INVALID", symbol)
    return {
        "Quantity_Multiple": multiple,
        "Minimum_Order_Quantity": minimum,
        "Odd_Lot_Sells_Verified": bool(rule.get("Odd_Lot_Sells_Verified", False)),
        "Odd_Lot_Unit": int(rule.get("Odd_Lot_Unit", 0)),
    }


def _sell_quantity(
    held_quantity: float, lot_rule: Mapping[str, Any]
) -> tuple[int, str]:
    if not np.isfinite(held_quantity) or held_quantity <= 0 or int(held_quantity) != held_quantity:
        raise PolicyBlockedError("HOLDING_QUANTITY_INVALID")
    quantity = int(held_quantity)
    regular = (
        quantity >= int(lot_rule["Minimum_Order_Quantity"])
        and quantity % int(lot_rule["Quantity_Multiple"]) == 0
    )
    if regular:
        return quantity, "REGULAR_LOT"
    odd_unit = int(lot_rule["Odd_Lot_Unit"])
    if (
        lot_rule["Odd_Lot_Sells_Verified"]
        and odd_unit > 0
        and quantity % odd_unit == 0
    ):
        return quantity, "VERIFIED_ODD_LOT"
    raise PolicyBlockedError("ODD_LOT_SELLABILITY_UNVERIFIED")


def _buy_quantity(
    budget: float, price: float, fee_rate: float, lot_rule: Mapping[str, Any]
) -> tuple[int, str]:
    if not finite_positive(price) or not np.isfinite(budget) or budget <= 0:
        return 0, "NO_DEPLOYABLE_BUDGET"
    quantity_multiple = int(lot_rule["Quantity_Multiple"])
    minimum = int(lot_rule["Minimum_Order_Quantity"])
    affordable = math.floor(budget / (price * (1.0 + fee_rate)))
    quantity = (affordable // quantity_multiple) * quantity_multiple
    if quantity < minimum:
        return 0, "CASH_BELOW_MINIMUM_ORDER"
    return quantity, "REGULAR_LOT"


def execute_orders(
    signals: pd.DataFrame,
    eligibility: pd.DataFrame,
    positions: Mapping[str, float],
    cash: float,
    portfolio_equity: float,
    board_lot_schedule: Mapping[str, Mapping[str, Any]],
    decision_date: Any,
    execution_date: Any,
    buy_open_provider: Callable[[str], Mapping[str, Any] | None],
    exit_open_quotes: Mapping[str, Mapping[str, Any] | None],
    allocation_function: Callable[[Sequence[str], float], AllocationResult],
    fee_rate: float,
    strategy_id: str,
    allocation_method: str,
    lineage: Mapping[str, str | None] | None = None,
) -> dict[str, Any]:
    decision_day = pd.Timestamp(decision_date).normalize()
    execution_day = pd.Timestamp(execution_date).normalize()
    if execution_day <= decision_day:
        raise NoLookaheadError("INVALID_EXECUTION_DATE")
    if not math.isclose(fee_rate, TRANSACTION_COST_RATE, rel_tol=0.0, abs_tol=1e-12):
        raise PolicyBlockedError("EXECUTION_FEE_POLICY_MISMATCH")
    if not np.isfinite(cash) or cash < 0 or not np.isfinite(portfolio_equity) or portfolio_equity < 0:
        raise ValueError("Cash and portfolio equity must be finite and nonnegative.")
    if "Symbol" not in signals or "Action" not in signals:
        raise PolicyBlockedError("SIGNAL_SCHEMA_INVALID")
    eligibility_map = (
        eligibility.set_index("Symbol").to_dict("index") if not eligibility.empty else {}
    )
    updated = {str(symbol).upper(): float(quantity) for symbol, quantity in positions.items()}
    current_cash = float(cash)
    rows: list[dict[str, Any]] = []
    sell_symbols = {
        str(row["Symbol"]).upper()
        for row in signals.to_dict("records")
        if row.get("Action") == "SELL"
    }
    sell_symbols.update(
        symbol
        for symbol, item in eligibility_map.items()
        if bool(item.get("Forced_Exit", False))
    )

    def transaction(
        symbol: str,
        side: str,
        quantity: int,
        price: float | None,
        status: str,
        reason: str,
        cash_delta: float,
        cash_after: float,
        allocated: AllocationResult | None = None,
        raw_open: float | None = None,
    ) -> dict[str, Any]:
        gross = float(quantity * price) if quantity and price is not None else 0.0
        fee = gross * TRANSACTION_COST_RATE if quantity and price is not None else 0.0
        slippage_cost = (
            float(quantity * (price - raw_open))
            if quantity and price is not None and raw_open is not None and side == "BUY"
            else float(quantity * (raw_open - price))
            if quantity and price is not None and raw_open is not None and side == "SELL"
            else np.nan
        )
        return {
            "Strategy_ID": strategy_id,
            "Allocation_Method": allocation_method,
            "Predictor_Date": decision_day,
            "Execution_Date": execution_day,
            "Symbol": symbol,
            "Side": side,
            "Quantity": int(quantity),
            "Raw_Open": float(raw_open) if raw_open is not None else np.nan,
            "Fill_Price": float(price) if price is not None else np.nan,
            "Gross_Notional": gross,
            "Fee_Rate": TRANSACTION_COST_RATE,
            "Fee": fee,
            "Slippage_Cost": slippage_cost,
            "Slippage_Status": (
                "COMPUTED_FROM_APPROVED_0.001"
                if np.isfinite(slippage_cost)
                else "NOT_APPLICABLE_NO_FILL"
            ),
            "Cash_Delta": cash_delta,
            "Cash_Debit_or_Credit": cash_delta,
            "Cash_After": cash_after,
            "Status": status,
            "Reason": reason,
            "Actual_Executed_Weight": (
                gross / allocated.deployable_cash
                if allocated is not None and allocated.deployable_cash > 0
                else np.nan
            ),
            "Allocation_Reason": allocated.reason if allocated else "",
            "Risk_Data_Hash": allocated.risk_data_hash if allocated else None,
            "Method_Input_Hash": allocated.method_input_hash if allocated else None,
            "Universe_Snapshot_ID": (lineage or {}).get("Universe_Snapshot_ID"),
            "Prediction_Snapshot_ID": (lineage or {}).get("Prediction_Snapshot_ID"),
            "Execution_Eligibility_Snapshot_ID": (lineage or {}).get(
                "Execution_Eligibility_Snapshot_ID"
            ),
            "Model_Version": (lineage or {}).get("Model_Version"),
            "Allocation_Policy_Version": (lineage or {}).get(
                "Allocation_Policy_Version"
            ),
        }

    for symbol in sorted(sell_symbols):
        held_quantity = updated.get(symbol, 0.0)
        if held_quantity <= 0:
            continue
        quote = exit_open_quotes.get(symbol)
        if not quote or not finite_positive(quote.get("Open")) or quote.get("Status") != "VERIFIED":
            rows.append(transaction(symbol, "SELL", 0, None, "NO_FILL", "MISSING_OR_UNVERIFIED_OPEN", 0.0, current_cash))
            continue
        try:
            rule = _lot_rule_for(symbol, execution_day, board_lot_schedule)
            quantity, lot_type = _sell_quantity(held_quantity, rule)
        except PolicyBlockedError as error:
            rows.append(transaction(symbol, "SELL", 0, None, "BLOCKED", error.reason, 0.0, current_cash))
            continue
        raw_open = float(quote["Open"])
        fill_price = raw_open * (1.0 - MAX_SLIPPAGE_RATE)
        gross = quantity * fill_price
        fee = gross * TRANSACTION_COST_RATE
        current_cash += gross - fee
        updated[symbol] = 0.0
        rows.append(
            transaction(
                symbol,
                "SELL",
                quantity,
                fill_price,
                "FILLED",
                lot_type,
                gross - fee,
                current_cash,
                raw_open=raw_open,
            )
        )

    buy_candidates: list[str] = []
    for row in signals.to_dict("records"):
        symbol = str(row["Symbol"]).upper()
        action = row.get("Action")
        eligibility_row = eligibility_map.get(symbol, {})
        requires_buy = action == "BUY"
        if requires_buy and eligibility_row.get("Execution_Eligibility") == "ELIGIBLE_POLICY":
            buy_candidates.append(symbol)
        elif requires_buy:
            rows.append(transaction(symbol, "BUY", 0, None, "BLOCKED", "UNIVERSE_OR_EVIDENCE_BLOCKED", 0.0, current_cash))
    buy_candidates = sorted(set(buy_candidates))
    allocation = allocation_function(buy_candidates, current_cash) if buy_candidates else AllocationResult(
        allocation_method,
        "NO_ELIGIBLE_BUYS",
        "NO_ELIGIBLE_BUY_SIGNALS",
        reserve_cash=current_cash,
        deployable_cash=current_cash,
    )
    if allocation.status == "ALLOCATED":
        for symbol in buy_candidates:
            try:
                rule = _lot_rule_for(symbol, execution_day, board_lot_schedule)
            except PolicyBlockedError as error:
                rows.append(transaction(symbol, "BUY", 0, None, "BLOCKED", error.reason, 0.0, current_cash, allocation))
                continue
            quote = buy_open_provider(symbol)
            if not quote or not finite_positive(quote.get("Open")) or quote.get("Status") != "VERIFIED":
                rows.append(transaction(symbol, "BUY", 0, None, "NO_FILL", "MISSING_OR_UNVERIFIED_OPEN", 0.0, current_cash, allocation))
                continue
            raw_open = float(quote["Open"])
            fill_price = raw_open * (1.0 + MAX_SLIPPAGE_RATE)
            budget = min(allocation.allocated_amounts.get(symbol, 0.0), current_cash)
            quantity, lot_type = _buy_quantity(
                budget, fill_price, TRANSACTION_COST_RATE, rule
            )
            if quantity <= 0:
                rows.append(
                    transaction(
                        symbol,
                        "BUY",
                        0,
                        fill_price,
                        "NO_FILL",
                        lot_type,
                        0.0,
                        current_cash,
                        allocation,
                        raw_open,
                    )
                )
                continue
            gross = quantity * fill_price
            fee = gross * TRANSACTION_COST_RATE
            if gross + fee > current_cash + 1e-8:
                raise PolicyBlockedError("CASH_CONSTRAINT_BREACHED", symbol)
            current_cash -= gross + fee
            updated[symbol] = updated.get(symbol, 0.0) + quantity
            rows.append(
                transaction(
                    symbol,
                    "BUY",
                    quantity,
                    fill_price,
                    "FILLED",
                    lot_type,
                    -(gross + fee),
                    current_cash,
                    allocation,
                    raw_open,
                )
            )
    else:
        for symbol in buy_candidates:
            rows.append(
                transaction(
                    symbol,
                    "BUY",
                    0,
                    None,
                    "NO_FILL",
                    f"ALLOCATION_{allocation.status}:{allocation.reason}",
                    0.0,
                    current_cash,
                    allocation,
                )
            )
    return {
        "Cash": current_cash,
        "Positions": updated,
        "Transactions": pd.DataFrame(rows, columns=TRANSACTION_COLUMNS),
        "Allocation": allocation,
    }


## ส่วนที่ 07 — Corporate actions และการประเมินมูลค่า

ไม่ใช้ event จาก vendor เป็นเงื่อนไข corporate action ที่ยืนยันแล้ว รายการจะถูกใช้เมื่อมี reconciliation สู่เงื่อนไขทางการครบถ้วนเท่านั้น ราคาปิดเดิมใช้ทำ valuation ได้ไม่เกินสาม verified exchange sessions หลังจากนั้น valuation ของพอร์ตเป็น unavailable และไม่สร้างผลตอบแทนศูนย์เทียม

In [9]:
def apply_corporate_actions(
    effective_date: Any,
    cash: float,
    positions: Mapping[str, float],
    corporate_action_ledger: pd.DataFrame,
    dividend_entitlements: Mapping[str, float],
    applied_event_ids: Iterable[str] = (),
) -> dict[str, Any]:
    day = pd.Timestamp(effective_date).normalize()
    updated = {str(symbol).upper(): float(quantity) for symbol, quantity in positions.items()}
    updated_cash = float(cash)
    applied = set(str(value) for value in applied_event_ids)
    if corporate_action_ledger.empty:
        return {"Cash": updated_cash, "Positions": updated, "Applied_Event_IDs": applied}
    required = {
        "Event_ID",
        "Symbol",
        "Event_Type",
        "Effective_Date",
        "Payment_Date",
        "Record_Date",
        "Cash_Per_Share",
        "Share_Ratio",
        "Reconciliation_Status",
    }
    if not required.issubset(corporate_action_ledger.columns):
        raise PolicyBlockedError("CORPORATE_ACTION_SCHEMA_INVALID")
    for event in corporate_action_ledger.to_dict("records"):
        event_id = str(event["Event_ID"])
        symbol = str(event["Symbol"]).upper()
        if event_id in applied:
            continue
        event_type = str(event["Event_Type"]).upper()
        if event_type in {"SPLIT", "STOCK_DIVIDEND"} and updated.get(symbol, 0.0) <= 0:
            continue
        event_date = event.get("Effective_Date") if event_type in {"SPLIT", "STOCK_DIVIDEND"} else event.get("Payment_Date")
        if pd.isna(event_date) or pd.Timestamp(event_date).normalize() != day:
            continue
        if event.get("Reconciliation_Status") != "VERIFIED_OFFICIAL_TERMS":
            raise PolicyBlockedError("CORPORATE_ACTION_NOT_RECONCILED", event_id)
        if event_type in {"SPLIT", "STOCK_DIVIDEND"}:
            ratio = float(event["Share_Ratio"])
            if not np.isfinite(ratio) or ratio <= 0:
                raise PolicyBlockedError("CORPORATE_ACTION_RATIO_INVALID", event_id)
            adjusted = updated[symbol] * ratio
            if int(adjusted) != adjusted:
                raise PolicyBlockedError("FRACTIONAL_CORPORATE_ACTION_SHARES_UNRESOLVED", event_id)
            updated[symbol] = adjusted
        elif event_type == "CASH_DIVIDEND":
            entitlement = dividend_entitlements.get(event_id)
            amount = float(event["Cash_Per_Share"])
            if entitlement is None or not np.isfinite(entitlement) or entitlement < 0:
                raise PolicyBlockedError("DIVIDEND_ENTITLEMENT_UNVERIFIED", event_id)
            if not np.isfinite(amount) or amount < 0:
                raise PolicyBlockedError("DIVIDEND_AMOUNT_INVALID", event_id)
            updated_cash += entitlement * amount
        else:
            raise PolicyBlockedError("CORPORATE_ACTION_TYPE_UNSUPPORTED", event_type)
        applied.add(event_id)
    return {"Cash": updated_cash, "Positions": updated, "Applied_Event_IDs": applied}


POSITION_VALUATION_COLUMNS = [
    "Date",
    "Symbol",
    "Quantity",
    "Mark_Price",
    "Price_Date",
    "Stale_Session_Age",
    "Valuation_Status",
    "Position_Value",
]


def mark_portfolio(
    date: Any,
    session_ordinal: int | None,
    calendar_verified: bool,
    cash: float,
    positions: Mapping[str, float],
    observed_closes: Mapping[str, Mapping[str, Any] | None],
    last_valid_marks: Mapping[str, Mapping[str, Any]],
) -> dict[str, Any]:
    day = pd.Timestamp(date).normalize()
    marks: dict[str, dict[str, Any]] = {
        str(symbol).upper(): dict(record) for symbol, record in last_valid_marks.items()
    }
    rows: list[dict[str, Any]] = []
    unavailable = not calendar_verified or session_ordinal is None
    stale_count = 0
    market_value = 0.0
    for symbol, raw_quantity in positions.items():
        symbol = str(symbol).upper()
        quantity = float(raw_quantity)
        quote = observed_closes.get(symbol)
        if not calendar_verified or session_ordinal is None:
            rows.append(
                {
                    "Date": day,
                    "Symbol": symbol,
                    "Quantity": quantity,
                    "Mark_Price": np.nan,
                    "Price_Date": None,
                    "Stale_Session_Age": None,
                    "Valuation_Status": "PORTFOLIO_VALUE_UNAVAILABLE",
                    "Position_Value": np.nan,
                }
            )
            unavailable = True
            continue
        if quote and quote.get("Status") == "VERIFIED" and finite_positive(quote.get("Close")):
            price = float(quote["Close"])
            marks[symbol] = {
                "Price": price,
                "Price_Date": day,
                "Session_Ordinal": int(session_ordinal),
            }
            age = 0
            status = "PORTFOLIO_VALUE_VALID"
        else:
            prior = marks.get(symbol)
            if prior is None:
                price, age, status = np.nan, None, "PORTFOLIO_VALUE_UNAVAILABLE"
                unavailable = True
            else:
                age = int(session_ordinal) - int(prior["Session_Ordinal"])
                price = float(prior["Price"])
                if 1 <= age <= MAX_STALE_VERIFIED_SESSIONS:
                    status = "PORTFOLIO_VALUE_STALE"
                    stale_count += 1
                else:
                    status = "PORTFOLIO_VALUE_UNAVAILABLE"
                    unavailable = True
        position_value = quantity * price if np.isfinite(price) else np.nan
        if np.isfinite(position_value):
            market_value += position_value
        rows.append(
            {
                "Date": day,
                "Symbol": symbol,
                "Quantity": quantity,
                "Mark_Price": price,
                "Price_Date": marks.get(symbol, {}).get("Price_Date"),
                "Stale_Session_Age": age,
                "Valuation_Status": status,
                "Position_Value": position_value,
            }
        )
    portfolio_status = (
        "PORTFOLIO_VALUE_UNAVAILABLE"
        if unavailable
        else "PORTFOLIO_VALUE_STALE"
        if stale_count > 0
        else "PORTFOLIO_VALUE_VALID"
    )
    equity = float(cash + market_value) if not unavailable and np.isfinite(cash) else np.nan
    return {
        "Daily": {
            "Date": day,
            "Cash": float(cash),
            "Market_Value": market_value if not unavailable else np.nan,
            "Portfolio_Equity": equity,
            "Portfolio_Equity_Status": portfolio_status,
            "Stale_Holding_Count": stale_count,
            "Open_Position_Count": sum(quantity > 0 for quantity in positions.values()),
            "Session_Ordinal": session_ordinal,
        },
        "Positions": pd.DataFrame(rows, columns=POSITION_VALUATION_COLUMNS),
        "Last_Valid_Marks": marks,
    }


## ส่วนที่ 08 — ผลตอบแทนและตัวชี้วัดรายช่วง

สร้าง segment ใหม่เมื่อ equity ไม่ valid หรือ session ordinal ขาดตอน ห้ามเชื่อมผลตอบแทนข้ามช่องว่าง ตาม E-004 ใช้ 252 verified exchange-session return intervals ต่อปี และ annualize ได้เมื่อมีอย่างน้อย 253 marks Risk-free Rate และ MAR แยกจาก E-004 และยังต้องรอการอนุมัติ

In [10]:
PERFORMANCE_COLUMNS = [
    "Segment_ID",
    "Start_Date",
    "End_Date",
    "Valid_Equity_Marks",
    "Valid_Return_Intervals",
    "Elapsed_Calendar_Days",
    "Total_Return",
    "Annualized_Return",
    "Annualized_Volatility",
    "Sharpe_Ratio",
    "Sortino_Ratio",
    "Maximum_Drawdown",
    "Calmar_Ratio",
    "Average_Cash",
    "Stale_Days",
    "Annualization_Status",
    "Risk_Free_Status",
    "MAR_Status",
    "Status",
]


def valid_equity_segments(daily_portfolio: pd.DataFrame) -> list[pd.DataFrame]:
    required = {
        "Date",
        "Session_Ordinal",
        "Portfolio_Equity",
        "Portfolio_Equity_Status",
    }
    if not required.issubset(daily_portfolio.columns):
        raise PolicyBlockedError("DAILY_PORTFOLIO_SCHEMA_INVALID")
    frame = daily_portfolio.copy()
    frame["Date"] = pd.to_datetime(frame["Date"]).dt.normalize()
    frame = frame.sort_values(["Session_Ordinal", "Date"], kind="stable")
    frame = frame.reset_index(drop=True)
    if frame.duplicated(["Date"]).any():
        raise PolicyBlockedError("DAILY_PORTFOLIO_DUPLICATE_DATE")
    segments: list[list[int]] = []
    current: list[int] = []
    previous_ordinal: int | None = None
    for index, row in frame.iterrows():
        valid = (
            row["Portfolio_Equity_Status"] == "PORTFOLIO_VALUE_VALID"
            and finite_positive(row["Portfolio_Equity"])
            and pd.notna(row["Session_Ordinal"])
        )
        ordinal = int(row["Session_Ordinal"]) if pd.notna(row["Session_Ordinal"]) else None
        if not valid:
            if current:
                segments.append(current)
                current = []
            previous_ordinal = None
            continue
        if current and (previous_ordinal is None or ordinal != previous_ordinal + 1):
            segments.append(current)
            current = []
        current.append(index)
        previous_ordinal = ordinal
    if current:
        segments.append(current)
    return [frame.loc[indices].reset_index(drop=True) for indices in segments]


def calculate_segment_metrics(
    daily_portfolio: pd.DataFrame,
    risk_free_rate: float | None = None,
    risk_free_rate_approved: bool = False,
    mar: float | None = None,
    mar_approved: bool = False,
) -> pd.DataFrame:
    if risk_free_rate is not None and not risk_free_rate_approved:
        raise PolicyBlockedError("RISK_FREE_RATE_NOT_APPROVED")
    if mar is not None and not mar_approved:
        raise PolicyBlockedError("MAR_NOT_APPROVED")
    if risk_free_rate is not None and risk_free_rate <= -1:
        raise ValueError("Risk-free rate must be greater than -100%.")
    if mar is not None and mar <= -1:
        raise ValueError("MAR must be greater than -100%.")
    segments = valid_equity_segments(daily_portfolio)
    records: list[dict[str, Any]] = []
    for segment_number, segment in enumerate(segments, start=1):
        equity = segment["Portfolio_Equity"].to_numpy(dtype=float)
        returns = equity[1:] / equity[:-1] - 1.0
        interval_count = len(returns)
        total_return = float(equity[-1] / equity[0] - 1.0) if len(equity) >= 2 else np.nan
        annualized_return = np.nan
        annualized_volatility = np.nan
        if interval_count >= ANNUALIZATION_INTERVALS and len(equity) >= MIN_ANNUALIZED_MARKS:
            annualized_return = float((equity[-1] / equity[0]) ** (ANNUALIZATION_INTERVALS / interval_count) - 1.0)
            annualized_volatility = float(np.std(returns, ddof=1) * np.sqrt(ANNUALIZATION_INTERVALS))
            annualization_status = "ANNUALIZED_252_VERIFIED_INTERVALS"
        else:
            annualization_status = "UNANNUALIZED_BELOW_252_INTERVALS"
        drawdown = equity / np.maximum.accumulate(equity) - 1.0
        maximum_drawdown = float(np.min(drawdown)) if len(drawdown) else np.nan
        if risk_free_rate_approved and risk_free_rate is not None and interval_count >= 2:
            daily_rf = (1.0 + risk_free_rate) ** (1.0 / ANNUALIZATION_INTERVALS) - 1.0
            excess = returns - daily_rf
            sharpe = (
                float(np.mean(excess) / np.std(excess, ddof=1) * np.sqrt(ANNUALIZATION_INTERVALS))
                if np.std(excess, ddof=1) > 0
                else np.nan
            )
            risk_free_status = "APPROVED_AND_COMPUTED"
        else:
            sharpe = np.nan
            risk_free_status = "NEEDS_RESEARCHER_APPROVAL"
        if mar_approved and mar is not None and interval_count > 0:
            daily_mar = (1.0 + mar) ** (1.0 / ANNUALIZATION_INTERVALS) - 1.0
            downside = np.minimum(returns - daily_mar, 0.0)
            downside_deviation = float(np.sqrt(np.mean(np.square(downside))))
            sortino = (
                float(np.mean(returns - daily_mar) / downside_deviation * np.sqrt(ANNUALIZATION_INTERVALS))
                if downside_deviation > 0
                else np.nan
            )
            mar_status = "APPROVED_AND_COMPUTED"
        else:
            sortino = np.nan
            mar_status = "NEEDS_RESEARCHER_APPROVAL"
        calmar = (
            float(annualized_return / abs(maximum_drawdown))
            if np.isfinite(annualized_return) and maximum_drawdown < 0
            else np.nan
        )
        start = pd.Timestamp(segment["Date"].iloc[0])
        end = pd.Timestamp(segment["Date"].iloc[-1])
        records.append(
            {
                "Segment_ID": f"SEGMENT_{segment_number:04d}",
                "Start_Date": start,
                "End_Date": end,
                "Valid_Equity_Marks": int(len(equity)),
                "Valid_Return_Intervals": int(interval_count),
                "Elapsed_Calendar_Days": int((end - start).days),
                "Total_Return": total_return,
                "Annualized_Return": annualized_return,
                "Annualized_Volatility": annualized_volatility,
                "Sharpe_Ratio": sharpe,
                "Sortino_Ratio": sortino,
                "Maximum_Drawdown": maximum_drawdown,
                "Calmar_Ratio": calmar,
                "Average_Cash": float(segment["Cash"].mean()) if "Cash" in segment else np.nan,
                "Stale_Days": int(segment.get("Stale_Holding_Count", pd.Series(dtype=float)).sum()),
                "Annualization_Status": annualization_status,
                "Risk_Free_Status": risk_free_status,
                "MAR_Status": mar_status,
                "Status": "VALID_SEGMENT",
            }
        )
    if not records:
        records.append(
            {
                "Segment_ID": None,
                "Start_Date": None,
                "End_Date": None,
                "Valid_Equity_Marks": 0,
                "Valid_Return_Intervals": 0,
                "Elapsed_Calendar_Days": None,
                "Total_Return": np.nan,
                "Annualized_Return": np.nan,
                "Annualized_Volatility": np.nan,
                "Sharpe_Ratio": np.nan,
                "Sortino_Ratio": np.nan,
                "Maximum_Drawdown": np.nan,
                "Calmar_Ratio": np.nan,
                "Average_Cash": np.nan,
                "Stale_Days": 0,
                "Annualization_Status": "NOT_AVAILABLE",
                "Risk_Free_Status": "NEEDS_RESEARCHER_APPROVAL",
                "MAR_Status": "NEEDS_RESEARCHER_APPROVAL",
                "Status": "UNAVAILABLE_NO_VALID_SEGMENT",
            }
        )
    return pd.DataFrame(records, columns=PERFORMANCE_COLUMNS)


## ส่วนที่ 09 — Lineage, audit exports และการทดสอบสังเคราะห์

Run manifest เก็บ E-029 ทั้งห้ารายการแยกจาก optimizer inputs และเก็บ `Risk_Data_Hash` แยกต่างหาก หากข้อมูลไม่ครบจะเป็น `PROVENANCE_INCOMPLETE` และห้าม export ผลย้อนหลัง การทดสอบด้านล่างใช้ข้อมูลสังเคราะห์เท่านั้น ไม่มีการเรียก historical runner หรือเขียนผล backtest

In [11]:
PREDICTION_COLUMNS = [
    "Feature_Date",
    "Target_Date",
    "Symbol",
    "Prediction",
    "Model_Version",
    "Prediction_Snapshot_ID",
    "Model_Artifact_SHA256",
    "Universe_Snapshot_ID",
    "Execution_Eligibility_Snapshot_ID",
    "Allocation_Policy_Version",
    "vol_z",
    "Probability_0",
    "Probability_1",
    "Probability_2",
]
ALLOCATION_COLUMNS = [
    "Date",
    "Strategy_ID",
    "Allocation_Method",
    "Symbol",
    "Raw_Weight",
    "Final_Weight",
    "Cap_Adjustment",
    "Risky_Sleeve_Weight",
    "Cash_Budget",
    "Allocation_Budget_Residual_Cash",
    "Post_Execution_Residual_Cash",
    "Risk_Window",
    "Observation_Count",
    "Risk_Data_Cutoff",
    "Method_Status",
    "Failure_Reason",
    "Solver_Metadata",
    "Risk_Data_Hash",
    "Method_Input_Hash",
    "Universe_Snapshot_ID",
    "Prediction_Snapshot_ID",
    "Execution_Eligibility_Snapshot_ID",
    "Model_Version",
    "Allocation_Policy_Version",
]
DAILY_PORTFOLIO_COLUMNS = [
    "Date",
    "Strategy_ID",
    "Allocation_Method",
    "Cash",
    "Market_Value",
    "Portfolio_Equity",
    "Portfolio_Equity_Status",
    "Stale_Holding_Count",
    "Open_Position_Count",
    "Session_Ordinal",
    "Risk_Data_Hash",
    "Universe_Snapshot_ID",
    "Prediction_Snapshot_ID",
    "Execution_Eligibility_Snapshot_ID",
    "Model_Version",
    "Allocation_Policy_Version",
]


def build_run_manifest(
    run_id: str,
    lineage: Mapping[str, str | None],
    risk_data_hash: str | None,
    risk_data_hash_by_decision: Mapping[str, str],
    source_file_hashes: Mapping[str, str],
    configuration: Mapping[str, Any],
    readiness: Mapping[str, Any],
) -> dict[str, Any]:
    missing = [key for key in E029_LINEAGE_FIELDS if not lineage.get(key)]
    if not risk_data_hash:
        missing.append("Risk_Data_Hash")
    status = "PROVENANCE_INCOMPLETE" if missing else "COMPLETE"
    return {
        "Run_ID": run_id or None,
        "Status": status,
        "Missing_Lineage_Fields": missing,
        "Universe_Snapshot_ID": lineage.get("Universe_Snapshot_ID"),
        "Prediction_Snapshot_ID": lineage.get("Prediction_Snapshot_ID"),
        "Execution_Eligibility_Snapshot_ID": lineage.get("Execution_Eligibility_Snapshot_ID"),
        "Model_Version": lineage.get("Model_Version"),
        "Allocation_Policy_Version": lineage.get("Allocation_Policy_Version"),
        "Risk_Data_Hash": risk_data_hash,
        "Risk_Data_Hash_By_Decision": dict(risk_data_hash_by_decision),
        "Source_File_SHA256": dict(source_file_hashes),
        "Configuration": dict(configuration),
        "Readiness": dict(readiness),
        "Research_Backtest_Gate": readiness.get(
            "research_backtest_gate", readiness.get("Research_Backtest_Gate", "BLOCKED")
        ),
        "Phase_F": readiness.get("phase_f", readiness.get("Phase_F", "DO_NOT_BEGIN")),
        "Created_By": "RESEARCHER_SUPPLIED_RUN_CONTEXT",
        "Historical_Lineage_Backfilled": False,
    }


def validate_run_context(context: Mapping[str, Any]) -> None:
    required = {
        "Research_Backtest_Gate": "AUTHORIZED",
        "Feature_Source_Status": "VERIFIED",
        "Model_Provenance_Status": "VERIFIED",
        "Execution_Price_Status": "VERIFIED",
        "Calendar_Status": "VERIFIED",
        "Membership_Status": "VERIFIED",
        "Corporate_Action_Status": "VERIFIED",
        "Board_Lot_Status": "VERIFIED",
    }
    failures = [
        f"{key}={context.get(key)!r}"
        for key, expected in required.items()
        if context.get(key) != expected
    ]
    if context.get("Phase_F", "DO_NOT_BEGIN") != "DO_NOT_BEGIN":
        failures.append("Phase_F must remain DO_NOT_BEGIN")
    lineage = context.get("Lineage", {})
    failures.extend(
        f"{field_name}=MISSING"
        for field_name in E029_LINEAGE_FIELDS
        if not lineage.get(field_name)
    )
    if not context.get("Risk_Data_Hash"):
        failures.append("Risk_Data_Hash=MISSING")
    if failures:
        raise PolicyBlockedError("PREFLIGHT_BLOCKED", "; ".join(failures))


def _records_for_date(frame: pd.DataFrame, date: pd.Timestamp) -> pd.DataFrame:
    copy = frame.copy()
    copy["Date"] = pd.to_datetime(copy["Date"]).dt.normalize()
    return copy.loc[copy["Date"].eq(date)].copy()


def run_portfolio_backtest(
    context: Mapping[str, Any],
    strategy_id: str,
    allocation_method: str,
) -> dict[str, Any]:
    validate_run_context(context)
    if strategy_id not in STRATEGIES or allocation_method not in ALLOCATION_METHODS:
        raise ValueError("Unknown strategy or allocation method.")
    sessions = pd.DatetimeIndex(pd.to_datetime(context["Verified_Sessions"])).normalize().unique().sort_values()
    if len(sessions) < 2:
        raise PolicyBlockedError("INSUFFICIENT_VERIFIED_SESSIONS")
    features = context["Feature_Data"].copy()
    features["Date"] = pd.to_datetime(features["Date"]).dt.normalize()
    execution_data = context["Execution_Data"].copy()
    returns = context["Returns"].copy()
    membership = context["Membership_Evidence"]
    exclusions = context.get("Exclusions", pd.DataFrame())
    board_lot_schedule = context["Board_Lot_Schedule"]
    fee_rate = float(context["Fee_Rate"])
    strategy_parameters = context["Strategy_Parameters"][strategy_id]
    model = context["Model"]
    lineage = dict(context["Lineage"])
    positions: dict[str, float] = {}
    cash = float(context["Initial_Cash"])
    equity = cash
    last_marks: dict[str, dict[str, Any]] = {}
    applied_event_ids: set[str] = set()
    dividend_entitlements = context.get("Dividend_Entitlements", {})
    prediction_rows: list[pd.DataFrame] = []
    eligibility_rows: list[pd.DataFrame] = []
    allocation_rows: list[dict[str, Any]] = []
    transaction_frames: list[pd.DataFrame] = []
    daily_rows: list[dict[str, Any]] = []
    valuation_frames: list[pd.DataFrame] = []
    risk_hash_by_decision: dict[str, str] = {}
    session_to_ordinal = {day: index for index, day in enumerate(sessions)}
    for predictor_date, execution_date in zip(sessions[:-1], sessions[1:]):
        feature_rows = features.loc[features["Date"].eq(predictor_date)].copy()
        if feature_rows.empty:
            continue
        prediction = generate_predictions(
            model=model,
            features=feature_rows,
            predictor_date=predictor_date,
            target_date=execution_date,
            feature_source_status=context["Feature_Source_Status"],
            model_metadata_match=context["Model_Metadata_Match"],
            lineage=lineage,
        )
        prediction_rows.append(prediction)
        current_weights: dict[str, float] = {}
        for symbol, quantity in positions.items():
            mark = last_marks.get(symbol)
            if mark and equity > 0:
                current_weights[symbol] = float(quantity * mark["Price"] / equity)
        signals = strategy_signals(
            prediction, current_weights, strategy_id, strategy_parameters
        )
        eligibility = build_dynamic_universe(
            prediction,
            predictor_date,
            execution_date,
            membership,
            exclusions,
            context["Identity_Status"],
            context["Corporate_Action_By_Symbol"],
            context["Price_Status_By_Symbol"],
            positions,
            lineage,
        )
        eligibility_rows.append(eligibility)
        corporate = apply_corporate_actions(
            execution_date,
            cash,
            positions,
            context["Corporate_Actions"],
            dividend_entitlements,
            applied_event_ids,
        )
        cash = corporate["Cash"]
        positions = corporate["Positions"]
        applied_event_ids = corporate["Applied_Event_IDs"]
        execution_rows = _records_for_date(execution_data, execution_date)
        quote_map = {
            str(row["Symbol"]).upper(): {"Open": row.get("Open"), "Status": row.get("Status")}
            for row in execution_rows.to_dict("records")
        }
        buy_candidates = [
            str(row["Symbol"]).upper()
            for row in signals.to_dict("records")
            if row["Action"] in {"BUY", "REBALANCE"}
        ]

        def allocation_function(
            symbols: Sequence[str], available_cash: float
        ) -> AllocationResult:
            if not symbols:
                return AllocationResult(
                    allocation_method,
                    "NO_ELIGIBLE_BUYS",
                    "NO_ELIGIBLE_BUY_SIGNALS",
                    reserve_cash=available_cash,
                )
            if allocation_method == "EW":
                return allocate_ew(symbols, available_cash)
            return allocate_method(
                allocation_method,
                symbols,
                available_cash,
                returns,
                predictor_date,
                sessions,
                True,
                context["Calendar_ID"],
            )

        def buy_open_provider(symbol: str) -> Mapping[str, Any] | None:
            return quote_map.get(symbol)

        exit_quotes = {
            symbol: quote_map.get(symbol)
            for symbol in set(positions).union(
                eligibility.loc[eligibility["Forced_Exit"], "Symbol"].astype(str).str.upper()
            )
        }
        execution = execute_orders(
            signals=signals,
            eligibility=eligibility,
            positions=positions,
            cash=cash,
            portfolio_equity=equity,
            board_lot_schedule=board_lot_schedule,
            decision_date=predictor_date,
            execution_date=execution_date,
            buy_open_provider=buy_open_provider,
            exit_open_quotes=exit_quotes,
            allocation_function=allocation_function,
            fee_rate=fee_rate,
            strategy_id=strategy_id,
            allocation_method=allocation_method,
            lineage=lineage,
        )
        cash = execution["Cash"]
        positions = execution["Positions"]
        allocation = execution["Allocation"]
        for symbol in buy_candidates:
            allocation_rows.append(
                {
                    "Date": predictor_date,
                    "Strategy_ID": strategy_id,
                    "Allocation_Method": allocation_method,
                    "Symbol": symbol,
                    "Raw_Weight": allocation.raw_weights.get(symbol, 0.0),
                    "Final_Weight": allocation.weights.get(symbol, 0.0),
                    "Cap_Adjustment": allocation.weights.get(symbol, 0.0)
                    - allocation.raw_weights.get(symbol, 0.0),
                    "Risky_Sleeve_Weight": allocation.risky_sleeve_weights.get(
                        symbol, np.nan
                    ),
                    "Cash_Budget": allocation.deployable_cash,
                    "Allocation_Budget_Residual_Cash": allocation.reserve_cash,
                    "Post_Execution_Residual_Cash": execution["Cash"],
                    "Risk_Window": allocation.lookback_observations,
                    "Observation_Count": allocation.lookback_observations,
                    "Risk_Data_Cutoff": allocation.risk_cutoff,
                    "Method_Status": allocation.status,
                    "Failure_Reason": allocation.reason,
                    "Solver_Metadata": allocation.solver_diagnostics,
                    "Risk_Data_Hash": allocation.risk_data_hash,
                    "Method_Input_Hash": allocation.method_input_hash,
                    "Universe_Snapshot_ID": lineage.get("Universe_Snapshot_ID"),
                    "Prediction_Snapshot_ID": lineage.get("Prediction_Snapshot_ID"),
                    "Execution_Eligibility_Snapshot_ID": lineage.get(
                        "Execution_Eligibility_Snapshot_ID"
                    ),
                    "Model_Version": lineage.get("Model_Version"),
                    "Allocation_Policy_Version": lineage.get(
                        "Allocation_Policy_Version"
                    ),
                }
            )
        if allocation.risk_data_hash:
            risk_hash_by_decision[predictor_date.date().isoformat()] = allocation.risk_data_hash
        transaction_frames.append(execution["Transactions"])
        closes = {
            str(row["Symbol"]).upper(): {"Close": row.get("Close"), "Status": row.get("Status")}
            for row in execution_rows.to_dict("records")
        }
        valuation = mark_portfolio(
            execution_date,
            session_to_ordinal[execution_date],
            True,
            cash,
            positions,
            closes,
            last_marks,
        )
        last_marks = valuation["Last_Valid_Marks"]
        daily = valuation["Daily"]
        daily.update(
            {
                "Strategy_ID": strategy_id,
                "Allocation_Method": allocation_method,
                "Risk_Data_Hash": allocation.risk_data_hash,
                "Universe_Snapshot_ID": lineage.get("Universe_Snapshot_ID"),
                "Prediction_Snapshot_ID": lineage.get("Prediction_Snapshot_ID"),
                "Execution_Eligibility_Snapshot_ID": lineage.get(
                    "Execution_Eligibility_Snapshot_ID"
                ),
                "Model_Version": lineage.get("Model_Version"),
                "Allocation_Policy_Version": lineage.get(
                    "Allocation_Policy_Version"
                ),
            }
        )
        daily_rows.append(daily)
        valuation_frames.append(valuation["Positions"])
        if daily["Portfolio_Equity_Status"] == "PORTFOLIO_VALUE_VALID":
            equity = float(daily["Portfolio_Equity"])
    daily_frame = pd.DataFrame(daily_rows)
    all_transactions = pd.concat(transaction_frames, ignore_index=True) if transaction_frames else pd.DataFrame(columns=TRANSACTION_COLUMNS)
    all_position_valuations = pd.concat(valuation_frames, ignore_index=True) if valuation_frames else pd.DataFrame(columns=POSITION_VALUATION_COLUMNS)
    performance = calculate_segment_metrics(daily_frame, transactions=all_transactions, position_valuation=all_position_valuations)
    run_risk_hash = context["Risk_Data_Hash"]
    manifest = build_run_manifest(
        run_id=str(context.get("Run_ID", "")),
        lineage=lineage,
        risk_data_hash=run_risk_hash,
        risk_data_hash_by_decision=risk_hash_by_decision,
        source_file_hashes=context["Source_File_SHA256"],
        configuration={
            "Strategy_ID": strategy_id,
            "Allocation_Method": allocation_method,
            "Annualization_Intervals": ANNUALIZATION_INTERVALS,
            "Maximum_Position_Weight": MAX_POSITION_WEIGHT,
            "Cash_Reserve_Policy": "RESIDUAL_CASH_ONLY_NO_FIXED_50_PERCENT_RESERVE",
            "Fee_Rate": fee_rate,
        },
        readiness=context,
    )
    if manifest["Status"] != "COMPLETE":
        raise PolicyBlockedError("PROVENANCE_INCOMPLETE", str(manifest["Missing_Lineage_Fields"]))
    return {
        "Predictions": pd.concat(prediction_rows, ignore_index=True) if prediction_rows else pd.DataFrame(columns=PREDICTION_COLUMNS),
        "Eligibility": pd.concat(eligibility_rows, ignore_index=True) if eligibility_rows else pd.DataFrame(),
        "Allocation_Weights": pd.DataFrame(allocation_rows, columns=ALLOCATION_COLUMNS),
        "Transactions": pd.concat(transaction_frames, ignore_index=True) if transaction_frames else pd.DataFrame(columns=TRANSACTION_COLUMNS),
        "Daily_Portfolio": daily_frame,
        "Position_Valuation": pd.concat(valuation_frames, ignore_index=True) if valuation_frames else pd.DataFrame(columns=POSITION_VALUATION_COLUMNS),
        "Performance_Summary": performance,
        "Run_Manifest": manifest,
    }


def export_run_outputs(
    outputs: Mapping[str, Any],
    run_id: str,
    research_backtest_gate: str,
    output_root: Path = OUTPUT_DIR / "runs",
) -> Path:
    manifest = outputs.get("Run_Manifest", {})
    if research_backtest_gate != "AUTHORIZED" or manifest.get("Status") != "COMPLETE":
        raise PolicyBlockedError("EXPORT_BLOCKED_BY_GATE_OR_PROVENANCE")
    if not run_id or Path(run_id).name != run_id or run_id in {".", ".."}:
        raise ValueError("Run_ID must be a non-empty path-safe identifier.")
    run_dir = output_root / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    for name in (
        "Predictions",
        "Eligibility",
        "Allocation_Weights",
        "Transactions",
        "Daily_Portfolio",
        "Position_Valuation",
        "Performance_Summary",
    ):
        outputs[name].to_csv(run_dir / f"{name.lower()}.csv", index=False)
    (run_dir / "run_manifest.json").write_text(
        json.dumps(manifest, indent=2, default=str), encoding="utf-8"
    )
    return run_dir


In [12]:
# การแก้ไขแบบเจาะจง: Strategy Matrix, CVaR ties, annualization, finite validation และ performance metrics
# ใช้กติกาที่อนุมัติแล้วเท่านั้น ไม่เปลี่ยน source data, model หรือ gate


def generate_predictions(
    model: Any,
    features: pd.DataFrame,
    predictor_date: Any,
    target_date: Any,
    feature_source_status: str,
    model_metadata_match: bool,
    lineage: Mapping[str, str],
) -> pd.DataFrame:
    predictor_day = pd.Timestamp(predictor_date).normalize()
    target_day = pd.Timestamp(target_date).normalize()
    if target_day <= predictor_day:
        raise NoLookaheadError("INVALID_TARGET_DATE", "Target date must follow feature date.")
    if feature_source_status != "VERIFIED":
        raise PolicyBlockedError("FEATURE_SOURCE_UNVERIFIED", feature_source_status)
    if not model_metadata_match:
        raise PolicyBlockedError("MODEL_PROVENANCE_INCONSISTENT")
    missing = [column for column in FEATURES if column not in features.columns]
    if missing:
        raise PolicyBlockedError("FEATURE_SCHEMA_INVALID", str(missing))
    for field_name in ("Model_Version", "Prediction_Snapshot_ID"):
        if not lineage.get(field_name):
            raise PolicyBlockedError("PROVENANCE_INCOMPLETE", field_name)
    if "Date" in features:
        dates = pd.to_datetime(features["Date"]).dt.normalize()
        if not dates.eq(predictor_day).all():
            raise PolicyBlockedError("FEATURE_DATE_MISMATCH")
    ordered = features.loc[:, FEATURES].copy()
    try:
        numeric = ordered.apply(pd.to_numeric, errors="raise").to_numpy(dtype=float)
    except Exception as error:
        raise PolicyBlockedError("FEATURE_VALUES_NON_NUMERIC", str(error)) from error
    if not np.isfinite(numeric).all():
        raise PolicyBlockedError("FEATURE_VALUES_NONFINITE")
    probabilities = np.asarray(model.predict_proba(ordered), dtype=float)
    classes = [int(value) for value in model.classes_]
    if probabilities.shape != (len(features), len(classes)):
        raise PolicyBlockedError("MODEL_OUTPUT_SCHEMA_INVALID")
    if not np.isfinite(probabilities).all():
        raise PolicyBlockedError("MODEL_OUTPUT_NONFINITE")
    if (probabilities < 0).any() or (probabilities > 1).any() or not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-6):
        raise PolicyBlockedError("MODEL_PROBABILITY_INVALID")
    results = []
    for index, (_, source) in enumerate(features.iterrows()):
        record = {
            "Feature_Date": predictor_day,
            "Target_Date": target_day,
            "Symbol": str(source["Symbol"]).upper() if "Symbol" in source else "",
            "Prediction": classes[int(np.argmax(probabilities[index]))],
            "Model_Version": lineage["Model_Version"],
            "Prediction_Snapshot_ID": lineage["Prediction_Snapshot_ID"],
            "Model_Artifact_SHA256": lineage.get("Model_Artifact_SHA256"),
            "vol_z": source.get("vol_z", np.nan),
            "Universe_Snapshot_ID": lineage.get("Universe_Snapshot_ID"),
            "Execution_Eligibility_Snapshot_ID": lineage.get("Execution_Eligibility_Snapshot_ID"),
            "Allocation_Policy_Version": lineage.get("Allocation_Policy_Version"),
        }
        for class_index, class_value in enumerate(classes):
            record[f"Probability_{class_value}"] = float(probabilities[index, class_index])
        results.append(record)
    return pd.DataFrame(results)


def strategy_signals(
    predictions: pd.DataFrame,
    current_weights: Mapping[str, float],
    strategy_id: str,
    parameters: Mapping[str, Any] | None = None,
) -> pd.DataFrame:
    """สร้างสัญญาณจาก 9 Cartesian strategies; Normal (class 2) เป็น HOLD เสมอ."""
    if strategy_id not in STRATEGY_ACTION_MATRIX:
        raise ValueError(f"Unknown strategy: {strategy_id}")
    required = {"Symbol", "Prediction"}
    if not required.issubset(predictions.columns):
        raise PolicyBlockedError("PREDICTION_SCHEMA_INVALID", str(sorted(required - set(predictions.columns))))
    frame = predictions.copy()
    frame["Symbol"] = frame["Symbol"].astype(str).str.upper()
    frame["Prediction"] = pd.to_numeric(frame["Prediction"], errors="raise").astype(int)
    if not frame["Prediction"].isin([0, 1, 2]).all():
        raise PolicyBlockedError("PREDICTION_CLASS_INVALID")
    if frame["Symbol"].duplicated().any():
        raise PolicyBlockedError("DUPLICATE_PREDICTION_SYMBOL")
    frame["Current_Weight"] = frame["Symbol"].map(current_weights).fillna(0.0).astype(float)
    if not np.isfinite(frame["Current_Weight"].to_numpy()).all() or (frame["Current_Weight"] < 0).any():
        raise PolicyBlockedError("CURRENT_WEIGHT_INVALID")
    action_map = STRATEGY_ACTION_MATRIX[strategy_id]
    frame["Action"] = frame["Prediction"].map(action_map)
    frame["Target_Weight"] = frame["Current_Weight"]
    frame["Signal_Reason"] = "STRATEGY_HOLD"
    buy = frame["Action"].eq("BUY")
    sell = frame["Action"].eq("SELL")
    frame.loc[buy, "Target_Weight"] = MAX_POSITION_WEIGHT
    frame.loc[sell, "Target_Weight"] = 0.0
    frame.loc[buy, "Signal_Reason"] = "CARTESIAN_ACTION_BUY"
    frame.loc[sell, "Signal_Reason"] = "CARTESIAN_ACTION_SELL"
    # ป้องกัน Normal Buy/Sell แม้มีข้อมูลผิดพลาดใน strategy mapping
    normal = frame["Prediction"].eq(2)
    frame.loc[normal, "Action"] = "HOLD"
    frame.loc[normal, "Target_Weight"] = frame.loc[normal, "Current_Weight"]
    frame.loc[normal, "Signal_Reason"] = "NORMAL_ALWAYS_HOLD"
    frame["Strategy_ID"] = strategy_id
    return frame


def fractional_tail_cvar(
    losses: Sequence[float], session_dates: Sequence[Any], alpha: float = 0.95
) -> dict[str, Any]:
    values = np.asarray(losses, dtype=float)
    dates = pd.to_datetime(list(session_dates))
    if len(values) != len(dates) or len(values) == 0 or not np.isfinite(values).all():
        raise PolicyBlockedError("CVAR_SCENARIOS_INVALID")
    if not 0.0 < alpha < 1.0:
        raise ValueError("CVaR alpha must be between zero and one.")
    tail_mass = (1.0 - alpha) * len(values)
    if tail_mass <= 0 or tail_mass > len(values):
        raise PolicyBlockedError("CVAR_TAIL_MASS_INVALID")
    order = sorted(range(len(values)), key=lambda index: (-values[index], dates[index]))
    cutoff_rank = int(math.ceil(tail_mass)) - 1
    boundary_loss = float(values[order[cutoff_rank]])
    strictly_worse = [index for index, value in enumerate(values) if value > boundary_loss]
    tied = [index for index, value in enumerate(values) if value == boundary_loss]
    remaining_mass = tail_mass - len(strictly_worse)
    if not tied or remaining_mass < -1e-12 or remaining_mass > len(tied) + 1e-12:
        raise PolicyBlockedError("CVAR_BOUNDARY_TIE_INVALID")
    per_tied_weight = remaining_mass / len(tied)
    weighted_tail_sum = float(values[strictly_worse].sum()) + float(values[tied].sum()) * per_tied_weight
    return {
        "CVaR_Objective": weighted_tail_sum / tail_mass,
        "Tail_Mass": tail_mass,
        "Tail_Cutoff_Loss": boundary_loss,
        "Tie_Group_Size": len(tied),
        "Strictly_Worse_Count": len(strictly_worse),
        "Fractional_Weight_Per_Tied_Observation": per_tied_weight,
        "Tail_Scenario_Order": [dates[index].date().isoformat() for index in order],
    }


# Performance columns เพิ่มเติมตามข้อกำหนดของโครงการ
PERFORMANCE_COLUMNS = [
    "Segment_ID", "Start_Date", "End_Date", "Valid_Equity_Marks",
    "Valid_Return_Intervals", "Elapsed_Calendar_Days", "Total_Return",
    "Annualized_Return", "Annualized_Volatility", "Sharpe_Ratio",
    "Sortino_Ratio", "Maximum_Drawdown", "Calmar_Ratio",
    "Average_Cash_Weight", "Transaction_Count", "Total_Transaction_Cost",
    "Total_Turnover", "Stale_Valuation_Count", "Maximum_Stale_Verified_Sessions",
    "Annualization_Status", "Risk_Free_Status", "MAR_Status", "Status",
]


def calculate_segment_metrics(
    daily_portfolio: pd.DataFrame,
    risk_free_rate: float | None = None,
    risk_free_rate_approved: bool = False,
    mar: float | None = None,
    mar_approved: bool = False,
    transactions: pd.DataFrame | None = None,
    position_valuation: pd.DataFrame | None = None,
) -> pd.DataFrame:
    if risk_free_rate is not None and not risk_free_rate_approved:
        raise PolicyBlockedError("RISK_FREE_RATE_NOT_APPROVED")
    if mar is not None and not mar_approved:
        raise PolicyBlockedError("MAR_NOT_APPROVED")
    if risk_free_rate is not None and risk_free_rate <= -1:
        raise ValueError("Risk-free rate must be greater than -100%.")
    if mar is not None and mar <= -1:
        raise ValueError("MAR must be greater than -100%.")
    segments = valid_equity_segments(daily_portfolio)
    records = []
    tx = transactions.copy() if transactions is not None else pd.DataFrame()
    if not tx.empty and "Execution_Date" in tx:
        tx["Execution_Date"] = pd.to_datetime(tx["Execution_Date"]).dt.normalize()
    pv = position_valuation.copy() if position_valuation is not None else pd.DataFrame()
    if not pv.empty and "Date" in pv:
        pv["Date"] = pd.to_datetime(pv["Date"]).dt.normalize()
    for segment_number, segment in enumerate(segments, start=1):
        equity = segment["Portfolio_Equity"].to_numpy(dtype=float)
        returns = equity[1:] / equity[:-1] - 1.0
        interval_count = len(returns)
        total_return = float(equity[-1] / equity[0] - 1.0) if len(equity) >= 2 else np.nan
        annualized = interval_count >= ANNUALIZATION_INTERVALS and len(equity) >= MIN_ANNUALIZED_MARKS
        annualized_return = float((equity[-1] / equity[0]) ** (ANNUALIZATION_INTERVALS / interval_count) - 1.0) if annualized else np.nan
        annualized_volatility = float(np.std(returns, ddof=1) * np.sqrt(ANNUALIZATION_INTERVALS)) if annualized else np.nan
        annualization_status = "ANNUALIZED_252_VERIFIED_INTERVALS" if annualized else "UNANNUALIZED_BELOW_252_INTERVALS"
        drawdown = equity / np.maximum.accumulate(equity) - 1.0
        maximum_drawdown = float(np.min(drawdown)) if len(drawdown) else np.nan
        # Sharpe/Sortino เป็น annualized metrics จึงต้องผ่าน E-004 เช่นเดียวกัน
        if annualized and risk_free_rate_approved and risk_free_rate is not None:
            daily_rf = (1.0 + risk_free_rate) ** (1.0 / ANNUALIZATION_INTERVALS) - 1.0
            excess = returns - daily_rf
            std_excess = float(np.std(excess, ddof=1))
            sharpe = float(np.mean(excess) / std_excess * np.sqrt(ANNUALIZATION_INTERVALS)) if std_excess > 0 else np.nan
            risk_free_status = "APPROVED_AND_COMPUTED"
        else:
            sharpe = np.nan
            risk_free_status = "NEEDS_RESEARCHER_APPROVAL" if not risk_free_rate_approved or risk_free_rate is None else "UNANNUALIZED_BELOW_252_INTERVALS"
        if annualized and mar_approved and mar is not None:
            daily_mar = (1.0 + mar) ** (1.0 / ANNUALIZATION_INTERVALS) - 1.0
            downside = np.minimum(returns - daily_mar, 0.0)
            downside_deviation = float(np.sqrt(np.mean(np.square(downside))))
            sortino = float(np.mean(returns - daily_mar) / downside_deviation * np.sqrt(ANNUALIZATION_INTERVALS)) if downside_deviation > 0 else np.nan
            mar_status = "APPROVED_AND_COMPUTED"
        else:
            sortino = np.nan
            mar_status = "NEEDS_RESEARCHER_APPROVAL" if not mar_approved or mar is None else "UNANNUALIZED_BELOW_252_INTERVALS"
        calmar = float(annualized_return / abs(maximum_drawdown)) if np.isfinite(annualized_return) and maximum_drawdown < 0 else np.nan
        start = pd.Timestamp(segment["Date"].iloc[0]).normalize()
        end = pd.Timestamp(segment["Date"].iloc[-1]).normalize()
        cash_weight = (segment["Cash"].astype(float) / segment["Portfolio_Equity"].astype(float)) if "Cash" in segment else pd.Series(dtype=float)
        average_cash_weight = float(cash_weight.replace([np.inf, -np.inf], np.nan).dropna().mean()) if not cash_weight.empty else np.nan
        if not tx.empty and {"Execution_Date", "Status"}.issubset(tx.columns):
            segment_tx = tx[tx["Execution_Date"].between(start, end) & tx["Status"].eq("FILLED")]
        else:
            segment_tx = pd.DataFrame()
        transaction_count = int(len(segment_tx))
        total_cost = float(segment_tx.get("Fee", pd.Series(dtype=float)).fillna(0).sum() + segment_tx.get("Slippage_Cost", pd.Series(dtype=float)).fillna(0).sum())
        gross_turnover = float(segment_tx.get("Gross_Notional", pd.Series(dtype=float)).fillna(0).sum())
        average_equity = float(np.mean(equity)) if len(equity) else np.nan
        total_turnover = gross_turnover / average_equity if average_equity and average_equity > 0 else np.nan
        stale_count = int(segment.get("Stale_Holding_Count", pd.Series(0, index=segment.index)).fillna(0).sum())
        if not pv.empty and {"Date", "Stale_Session_Age"}.issubset(pv.columns):
            segment_pv = pv[pv["Date"].between(start, end)]
            max_stale = float(pd.to_numeric(segment_pv["Stale_Session_Age"], errors="coerce").max()) if not segment_pv.empty and pd.to_numeric(segment_pv["Stale_Session_Age"], errors="coerce").notna().any() else np.nan
        else:
            max_stale = np.nan
        records.append({
            "Segment_ID": f"SEGMENT_{segment_number:04d}", "Start_Date": start, "End_Date": end,
            "Valid_Equity_Marks": int(len(equity)), "Valid_Return_Intervals": int(interval_count),
            "Elapsed_Calendar_Days": int((end-start).days), "Total_Return": total_return,
            "Annualized_Return": annualized_return, "Annualized_Volatility": annualized_volatility,
            "Sharpe_Ratio": sharpe, "Sortino_Ratio": sortino, "Maximum_Drawdown": maximum_drawdown,
            "Calmar_Ratio": calmar, "Average_Cash_Weight": average_cash_weight,
            "Transaction_Count": transaction_count, "Total_Transaction_Cost": total_cost,
            "Total_Turnover": total_turnover, "Stale_Valuation_Count": stale_count,
            "Maximum_Stale_Verified_Sessions": max_stale, "Annualization_Status": annualization_status,
            "Risk_Free_Status": risk_free_status, "MAR_Status": mar_status, "Status": "VALID_SEGMENT",
        })
    if not records:
        records.append({column: np.nan for column in PERFORMANCE_COLUMNS})
        records[-1].update({"Segment_ID": None, "Valid_Equity_Marks": 0, "Valid_Return_Intervals": 0, "Stale_Valuation_Count": 0, "Annualization_Status": "NOT_AVAILABLE", "Risk_Free_Status": "NEEDS_RESEARCHER_APPROVAL", "MAR_Status": "NEEDS_RESEARCHER_APPROVAL", "Status": "UNAVAILABLE_NO_VALID_SEGMENT"})
    return pd.DataFrame(records, columns=PERFORMANCE_COLUMNS)



### การทดสอบเชิงสังเคราะห์เท่านั้น

In [13]:
import tempfile


def _synthetic_membership(
    predictor_member: bool = True, execution_member: bool = True
) -> pd.DataFrame:
    base = [f"S{index:03d}" for index in range(49)]
    first = base + (["X"] if predictor_member else ["AUX1"])
    second = base + (["X"] if execution_member else ["AUX2"])
    rows = []
    for period, effective, symbols in (
        ("2026_H1", "2026-01-01", first),
        ("2026_H2", "2026-07-01", second),
    ):
        for symbol in symbols:
            rows.append(
                {
                    "Period": period,
                    "Effective_Date": effective,
                    "Symbol": symbol,
                    "Local_Official_Match": True,
                    "Status": "VERIFIED_FROM_OFFICIAL_SET_PDF",
                }
            )
    return pd.DataFrame(rows)


def _synthetic_returns(
    count: int, symbols: Sequence[str] = ("A", "B"), seed: int = 17
) -> tuple[pd.DataFrame, pd.DatetimeIndex]:
    dates = pd.date_range("2020-01-01", periods=count, freq="D")
    generator = np.random.default_rng(seed)
    values = generator.normal(0.0005, 0.01, size=(count, len(symbols)))
    rows = [
        {"Date": date, "Symbol": symbol, "Return": values[row_index, column_index]}
        for row_index, date in enumerate(dates)
        for column_index, symbol in enumerate(symbols)
    ]
    return pd.DataFrame(rows), dates


def _lot_schedule(
    symbol: str = "X", multiple: int = 10, minimum: int = 10, status: str = "VERIFIED"
) -> dict[str, dict[str, Any]]:
    return {
        symbol: {
            "Status": status,
            "Effective_From": "2020-01-01",
            "Effective_To": "2030-12-31",
            "Quantity_Multiple": multiple,
            "Minimum_Order_Quantity": minimum,
            "Odd_Lot_Sells_Verified": False,
            "Odd_Lot_Unit": 0,
        }
    }


def run_synthetic_tests() -> pd.DataFrame:
    results: list[dict[str, str]] = []

    def case(name: str, action: Callable[[], None]) -> None:
        try:
            action()
            results.append({"Case": name, "Status": "PASS", "Details": ""})
        except Exception as error:
            results.append(
                {"Case": name, "Status": "FAIL", "Details": f"{type(error).__name__}: {error}"}
            )

    verified = _synthetic_membership()
    predictions_x = pd.DataFrame([{"Symbol": "X", "Prediction": 1}])
    all_verified = {"X": "VERIFIED"}

    def assert_membership_scope() -> None:
        snapshot = membership_snapshot_as_of(verified, "2026-03-01")
        assert snapshot["status"] == "VERIFIED" and len(snapshot["members"]) == 50
        assert membership_snapshot_as_of(verified, "2019-01-01")["status"] == "UNKNOWN"

    case("01_membership_snapshot_scope", assert_membership_scope)

    def assert_membership_change_blocks_new_buy() -> None:
        evidence = _synthetic_membership(True, False)
        row = build_dynamic_universe(
            predictions_x,
            "2026-06-30",
            "2026-07-01",
            evidence,
            pd.DataFrame(),
            all_verified,
            all_verified,
            all_verified,
        ).iloc[0]
        assert row["Execution_Eligibility"] == "BLOCKED"
        assert not row["Forced_Exit"]

    case("02_membership_change_between_dates", assert_membership_change_blocks_new_buy)

    def assert_verified_removal_forces_exit() -> None:
        evidence = _synthetic_membership(True, False)
        row = build_dynamic_universe(
            predictions_x,
            "2026-06-30",
            "2026-07-01",
            evidence,
            pd.DataFrame(),
            all_verified,
            all_verified,
            all_verified,
            {"X": 10},
        ).iloc[0]
        assert row["Forced_Exit"] and row["Execution_Eligibility"] == "FORCED_EXIT_ONLY"

    case("03_verified_membership_removal", assert_verified_removal_forces_exit)

    def assert_held_symbol_missing_prediction_is_retained() -> None:
        row = build_dynamic_universe(
            pd.DataFrame(columns=["Symbol", "Prediction"]),
            "2026-06-30",
            "2026-07-01",
            _synthetic_membership(True, False),
            pd.DataFrame(),
            all_verified,
            all_verified,
            all_verified,
            {"X": 10},
        ).iloc[0]
        assert row["Symbol"] == "X" and row["Forced_Exit"]

    case("04_held_symbol_without_prediction", assert_held_symbol_missing_prediction_is_retained)

    def assert_unknown_membership_fails_closed() -> None:
        invalid = verified.copy()
        invalid.loc[:, "Status"] = "UNVERIFIED"
        output = build_dynamic_universe(
            predictions_x,
            "2026-03-01",
            "2026-03-02",
            invalid,
            pd.DataFrame(),
            all_verified,
            all_verified,
            all_verified,
        )
        assert output.iloc[0]["Execution_Eligibility"] == "BLOCKED"

    case("05_unverified_membership_fails_closed", assert_unknown_membership_fails_closed)

    def assert_exclusion_blocks_new_buy() -> None:
        exclusion = pd.DataFrame(
            [
                {
                    "Symbol": "X",
                    "Exclusion_Start_Date": "2026-01-01",
                    "Exclusion_End_Date": "2026-12-31",
                    "Decision_Status": "APPROVED_RESEARCH_EXCLUSION",
                }
            ]
        )
        output = build_dynamic_universe(
            predictions_x,
            "2026-03-01",
            "2026-03-02",
            verified,
            exclusion,
            all_verified,
            all_verified,
            all_verified,
        )
        assert output.iloc[0]["Eligibility_Reason"] == "APPROVED_RESEARCH_EXCLUSION"

    case("06_approved_exclusion_blocks_buy", assert_exclusion_blocks_new_buy)

    def assert_security_identity_required() -> None:
        output = build_dynamic_universe(
            predictions_x,
            "2026-03-01",
            "2026-03-02",
            verified,
            pd.DataFrame(),
            {},
            all_verified,
            all_verified,
        )
        assert output.iloc[0]["Eligibility_Reason"] == "SECURITY_IDENTITY_NOT_VERIFIED"

    case("07_security_identity_required", assert_security_identity_required)

    def assert_action_reconciliation_required() -> None:
        output = build_dynamic_universe(
            predictions_x,
            "2026-03-01",
            "2026-03-02",
            verified,
            pd.DataFrame(),
            all_verified,
            {},
            all_verified,
        )
        assert output.iloc[0]["Eligibility_Reason"] == "CORPORATE_ACTION_NOT_RECONCILED"

    case("08_corporate_action_reconciliation_required", assert_action_reconciliation_required)

    def assert_execution_price_eligibility_required() -> None:
        output = build_dynamic_universe(
            predictions_x,
            "2026-03-01",
            "2026-03-02",
            verified,
            pd.DataFrame(),
            all_verified,
            all_verified,
            {},
        )
        assert output.iloc[0]["Eligibility_Reason"] == "EXECUTION_PRICE_NOT_VERIFIED"

    case("09_execution_price_eligibility_required", assert_execution_price_eligibility_required)

    def assert_predictor_date_order() -> None:
        try:
            build_dynamic_universe(
                predictions_x,
                "2026-03-02",
                "2026-03-01",
                verified,
                pd.DataFrame(),
                all_verified,
                all_verified,
                all_verified,
            )
        except NoLookaheadError:
            return
        raise AssertionError("Invalid date order was accepted.")

    case("10_prediction_execution_date_order", assert_predictor_date_order)

    class FakeModel:
        classes_ = np.array([0, 1, 2])

        def __init__(self) -> None:
            self.seen_columns: list[str] = []

        def predict_proba(self, matrix: pd.DataFrame) -> np.ndarray:
            self.seen_columns = list(matrix.columns)
            return np.tile(np.array([[0.2, 0.3, 0.5]]), (len(matrix), 1))

    model = FakeModel()
    features = pd.DataFrame(
        [{**{name: float(index + 1) for index, name in enumerate(FEATURES)}, "Date": "2026-03-01", "Symbol": "X", "Target": 2}]
    )
    lineage = {
        "Universe_Snapshot_ID": "U1",
        "Prediction_Snapshot_ID": "P1",
        "Execution_Eligibility_Snapshot_ID": "E1",
        "Model_Version": "M1",
        "Allocation_Policy_Version": "A1",
    }

    def assert_unverified_feature_source_blocks_inference() -> None:
        try:
            generate_predictions(model, features, "2026-03-01", "2026-03-02", "UNKNOWN", True, lineage)
        except PolicyBlockedError as error:
            assert error.reason == "FEATURE_SOURCE_UNVERIFIED"
            return
        raise AssertionError("Unverified source was accepted.")

    case("11_feature_source_identity_required", assert_unverified_feature_source_blocks_inference)

    def assert_model_metadata_mismatch_blocks_inference() -> None:
        try:
            generate_predictions(model, features, "2026-03-01", "2026-03-02", "VERIFIED", False, lineage)
        except PolicyBlockedError as error:
            assert error.reason == "MODEL_PROVENANCE_INCONSISTENT"
            return
        raise AssertionError("Inconsistent model metadata was accepted.")

    case("12_model_metadata_consistency_required", assert_model_metadata_mismatch_blocks_inference)

    def assert_model_receives_only_approved_feature_order() -> None:
        result = generate_predictions(
            model, features, "2026-03-01", "2026-03-02", "VERIFIED", True, lineage
        )
        assert model.seen_columns == FEATURES
        assert "Target" not in model.seen_columns
        assert result.iloc[0]["Prediction_Snapshot_ID"] == "P1"
        parameters = {
            "entry_class": 2,
            "exit_class": 0,
            "entry_threshold": 0.4,
            "exit_threshold": 0.4,
            "top_k": 1,
        }
        for strategy in STRATEGIES:
            signals = strategy_signals(result, {}, strategy, parameters)
            assert signals.iloc[0]["Strategy_ID"] == strategy
            assert signals.iloc[0]["Action"] in {"BUY", "SELL", "HOLD"}
            assert signals.iloc[0]["Target_Weight"] <= MAX_POSITION_WEIGHT

    case("13_no_label_leakage_and_nine_strategy_policies", assert_model_receives_only_approved_feature_order)

    def assert_missing_feature_is_blocked() -> None:
        try:
            generate_predictions(
                model,
                features.drop(columns=[FEATURES[0]]),
                "2026-03-01",
                "2026-03-02",
                "VERIFIED",
                True,
                lineage,
            )
        except PolicyBlockedError as error:
            assert error.reason == "FEATURE_SCHEMA_INVALID"
            return
        raise AssertionError("Missing feature was accepted.")

    case("14_feature_schema_required", assert_missing_feature_is_blocked)

    def assert_manifest_gate_and_e029_lineage() -> None:
        incomplete = build_run_manifest("R1", {}, None, {}, {}, {}, {})
        assert incomplete["Status"] == "PROVENANCE_INCOMPLETE"
        with tempfile.TemporaryDirectory() as temporary:
            try:
                export_run_outputs({"Run_Manifest": incomplete}, "R1", "BLOCKED", Path(temporary))
            except PolicyBlockedError:
                assert list(Path(temporary).iterdir()) == []
                return
        raise AssertionError("Blocked export created files.")

    case("15_gate_blocks_export_and_missing_lineage", assert_manifest_gate_and_e029_lineage)

    def assert_all_lineage_fields_recorded() -> None:
        manifest = build_run_manifest("R1", lineage, "risk-hash", {"2026-03-01": "risk-hash"}, {}, {}, {})
        assert manifest["Status"] == "COMPLETE"
        assert all(manifest[key] == lineage[key] for key in E029_LINEAGE_FIELDS)
        assert manifest["Risk_Data_Hash"] == "risk-hash"

    case("16_e029_identifiers_and_risk_hash", assert_all_lineage_fields_recorded)

    def assert_ew_cash_reserve_and_position_cap() -> None:
        expected = {1: 0.25, 2: 0.5, 3: 0.75, 4: 1.0, 5: 1.0}
        for count, expected_investment in expected.items():
            result = allocate_ew([f"S{index}" for index in range(count)], 1000)
            assert result.status == "ALLOCATED"
            assert all(weight <= 0.25 + 1e-12 for weight in result.weights.values())
            assert abs(sum(result.weights.values()) - expected_investment) < 1e-12
            assert abs(result.reserve_cash - 1000 * (1.0 - expected_investment)) < 1e-8

    case("17_ew_position_cap_and_cash_reserve", assert_ew_cash_reserve_and_position_cap)

    def assert_common_dates_drop_incomplete_without_fill() -> None:
        returns_frame, dates = _synthetic_returns(4)
        returns_frame = returns_frame.loc[
            ~((returns_frame["Date"] == dates[1]) & (returns_frame["Symbol"] == "B"))
        ]
        panel, _, dropped = build_common_risk_panel(
            returns_frame, ["A", "B"], dates[-1] + pd.Timedelta(days=1), dates, True, "CAL1"
        )
        assert dropped == 1 and dates[1] not in panel.index

    case("18_common_complete_dates_no_imputation", assert_common_dates_drop_incomplete_without_fill)

    def assert_unverified_calendar_blocks_risk_panel() -> None:
        returns_frame, dates = _synthetic_returns(60)
        try:
            build_common_risk_panel(
                returns_frame, ["A", "B"], dates[-1] + pd.Timedelta(days=1), dates, False, None
            )
        except PolicyBlockedError as error:
            assert error.reason == "CALENDAR_NOT_VERIFIED"
            return
        raise AssertionError("Unverified calendar was accepted.")

    case("19_verified_calendar_required_for_risk", assert_unverified_calendar_blocks_risk_panel)

    def assert_iv_minimum_observation_boundary() -> None:
        under, under_dates = _synthetic_returns(59)
        blocked = allocate_method("IV", ["A", "B"], 1000, under, under_dates[-1] + pd.Timedelta(days=1), under_dates, True, "CAL1")
        assert blocked.status == "UNAVAILABLE" and "INSUFFICIENT_COMMON_OBSERVATIONS" in blocked.reason
        sufficient, sufficient_dates = _synthetic_returns(60)
        result = allocate_method("IV", ["A", "B"], 1000, sufficient, sufficient_dates[-1] + pd.Timedelta(days=1), sufficient_dates, True, "CAL1")
        assert result.status == "ALLOCATED" and result.lookback_observations == 60

    case("20_iv_sixty_observation_boundary", assert_iv_minimum_observation_boundary)

    def assert_erc_common_sample_and_rc() -> None:
        dates = pd.date_range("2020-01-01", periods=60, freq="D")
        x = np.sin(np.arange(60) * 2.0 * np.pi / 12.0) * 0.01
        y = np.cos(np.arange(60) * 2.0 * np.pi / 12.0) * 0.01
        returns_frame = pd.DataFrame(
            [{"Date": date, "Symbol": symbol, "Return": value} for date, a, b in zip(dates, x, y) for symbol, value in (("A", a), ("B", b))]
        )
        result = allocate_method("ERC", ["A", "B"], 1000, returns_frame, dates[-1] + pd.Timedelta(days=1), dates, True, "CAL1")
        assert result.status == "ALLOCATED"
        assert max(result.weights.values()) <= 0.25 + 1e-12
        assert abs(sum(result.weights.values()) - 0.5) < 1e-6

    case("21_erc_common_sample_and_contribution_constraints", assert_erc_common_sample_and_rc)

    def assert_hrp_constraints() -> None:
        returns_frame, dates = _synthetic_returns(60, ("A", "B", "C"))
        result = allocate_method("HRP", ["A", "B", "C"], 1000, returns_frame, dates[-1] + pd.Timedelta(days=1), dates, True, "CAL1")
        assert result.status == "ALLOCATED"
        assert len(result.weights) == 3 and max(result.weights.values()) <= 0.25 + 1e-12

    case("22_hrp_common_sample_and_caps", assert_hrp_constraints)

    def assert_minvar_constraints() -> None:
        returns_frame, dates = _synthetic_returns(60)
        result = allocate_method("MinVar", ["A", "B"], 1000, returns_frame, dates[-1] + pd.Timedelta(days=1), dates, True, "CAL1")
        assert result.status == "ALLOCATED"
        assert abs(sum(result.weights.values()) - 0.5) < 1e-6
        assert max(result.weights.values()) <= 0.25 + 1e-12

    case("23_minvar_bounds_and_cash_reserve", assert_minvar_constraints)

    def assert_cvar_252_boundary() -> None:
        under, under_dates = _synthetic_returns(251)
        blocked = allocate_method("CVaR", ["A", "B"], 1000, under, under_dates[-1] + pd.Timedelta(days=1), under_dates, True, "CAL1")
        assert blocked.status == "UNAVAILABLE"
        sufficient, sufficient_dates = _synthetic_returns(252)
        result = allocate_method("CVaR", ["A", "B"], 1000, sufficient, sufficient_dates[-1] + pd.Timedelta(days=1), sufficient_dates, True, "CAL1")
        assert result.status == "ALLOCATED" and result.lookback_observations == 252
        assert abs(sum(result.weights.values()) - 0.5) < 1e-6

    case("24_cvar_252_observation_boundary", assert_cvar_252_boundary)

    def assert_mpt_bounds_and_common_sample() -> None:
        returns_frame, dates = _synthetic_returns(60, ("A", "B", "C"))
        result = allocate_method("MPT", ["A", "B", "C"], 1000, returns_frame, dates[-1] + pd.Timedelta(days=1), dates, True, "CAL1")
        assert result.status == "ALLOCATED"
        assert sum(result.weights.values()) <= 1.0 + 1e-8
        assert max(result.weights.values()) <= 0.25 + 1e-12
        negative = returns_frame.copy()
        negative["Return"] = negative["Return"] - 0.1
        cash_result = allocate_method("MPT", ["A", "B", "C"], 1000, negative, dates[-1] + pd.Timedelta(days=1), dates, True, "CAL1")
        assert cash_result.status == "ALLOCATED"
        assert cash_result.reason == "OPTIMAL_ALL_CASH"
        assert cash_result.reserve_cash == 1000

    case("25_mpt_bounds_and_common_sample", assert_mpt_bounds_and_common_sample)

    def execution_fixture(
        quote: Mapping[str, Any] | None = None,
        schedule: Mapping[str, Mapping[str, Any]] | None = None,
        cash: float = 100.0,
        equity: float = 100.0,
        allocation_callback: Callable[[Sequence[str], float, float], AllocationResult] | None = None,
        positions: Mapping[str, float] | None = None,
        signals_frame: pd.DataFrame | None = None,
        eligibility_frame: pd.DataFrame | None = None,
        open_provider: Callable[[str], Mapping[str, Any] | None] | None = None,
    ) -> dict[str, Any]:
        default_signals = pd.DataFrame([{"Symbol": "X", "Action": "BUY"}])
        default_eligibility = pd.DataFrame(
            [{"Symbol": "X", "Execution_Eligibility": "ELIGIBLE_POLICY", "Forced_Exit": False}]
        )
        return execute_orders(
            signals=signals_frame if signals_frame is not None else default_signals,
            eligibility=eligibility_frame if eligibility_frame is not None else default_eligibility,
            positions=positions or {},
            cash=cash,
            portfolio_equity=equity,
            board_lot_schedule=schedule if schedule is not None else _lot_schedule(),
            decision_date="2026-03-01",
            execution_date="2026-03-02",
            buy_open_provider=open_provider or (lambda symbol: quote if quote is not None else {"Open": 1.0, "Status": "VERIFIED"}),
            exit_open_quotes={},
            allocation_function=            allocation_callback or (lambda symbols, available: allocate_ew(symbols, available)),
            fee_rate=TRANSACTION_COST_RATE,
            strategy_id="S1_Threshold",
            allocation_method="EW",
        )

    def assert_fee_accounting_and_lot_size() -> None:
        result = execution_fixture()
        filled = result["Transactions"].query("Status == 'FILLED'").iloc[0]
        assert filled["Quantity"] % 10 == 0
        assert abs(filled["Cash_Delta"] + filled["Gross_Notional"] + filled["Fee"]) < 1e-9
        assert result["Cash"] >= 0
        assert filled["Raw_Open"] == 1.0 and filled["Fill_Price"] > filled["Raw_Open"]
        assert filled["Slippage_Cost"] > 0

    case("26_buy_lot_fee_and_cash_accounting", assert_fee_accounting_and_lot_size)

    def assert_missing_open_no_fill() -> None:
        result = execution_fixture(quote=None, open_provider=lambda symbol: None)
        assert not (result["Transactions"]["Status"] == "FILLED").any()
        assert result["Cash"] == 100

    case("27_missing_open_preserves_cash", assert_missing_open_no_fill)

    def assert_minimum_order_no_fill() -> None:
        result = execution_fixture(
            quote={"Open": 100.0, "Status": "VERIFIED"},
            schedule=_lot_schedule(minimum=100, multiple=100),
        )
        assert result["Transactions"].iloc[0]["Status"] == "NO_FILL"

    case("28_minimum_order_size_enforced", assert_minimum_order_no_fill)

    def assert_allocation_precedes_open_observation() -> None:
        events: list[str] = []
        execution_fixture(
            allocation_callback=lambda symbols, available: (
                events.append("allocation") or allocate_ew(symbols, available)
            ),
            open_provider=lambda symbol: (events.append("open") or {"Open": 1.0, "Status": "VERIFIED"}),
        )
        assert events == ["allocation", "open"]

    case("29_allocation_precedes_execution_open", assert_allocation_precedes_open_observation)

    def assert_sell_before_buy() -> None:
        signals_frame = pd.DataFrame([{"Symbol": "Y", "Action": "BUY"}, {"Symbol": "X", "Action": "HOLD"}])
        eligibility_frame = pd.DataFrame(
            [
                {"Symbol": "Y", "Execution_Eligibility": "ELIGIBLE_POLICY", "Forced_Exit": False},
                {"Symbol": "X", "Execution_Eligibility": "FORCED_EXIT_ONLY", "Forced_Exit": True},
            ]
        )
        result = execute_orders(
            signals_frame,
            eligibility_frame,
            {"X": 10},
            100,
            200,
            {**_lot_schedule("X"), **_lot_schedule("Y")},
            "2026-03-01",
            "2026-03-02",
            lambda symbol: {"Open": 1.0, "Status": "VERIFIED"},
            {"X": {"Open": 10.0, "Status": "VERIFIED"}},
            lambda symbols, available: allocate_ew(symbols, available),
            TRANSACTION_COST_RATE,
            "S1_Threshold",
            "EW",
        )
        filled = result["Transactions"].query("Status == 'FILLED'")
        assert list(filled["Side"]) == ["SELL", "BUY"]

    case("30_sell_orders_precede_buy_orders", assert_sell_before_buy)

    def assert_unverified_lot_rule_blocks_before_open() -> None:
        observed: list[str] = []
        result = execution_fixture(
            schedule=_lot_schedule(status="BLOCKED"),
            open_provider=lambda symbol: (observed.append(symbol) or {"Open": 1.0, "Status": "VERIFIED"}),
        )
        assert result["Transactions"].iloc[0]["Status"] == "BLOCKED"
        assert observed == []

    case("31_unverified_board_lot_blocks_sizing", assert_unverified_lot_rule_blocks_before_open)

    def assert_unreconciled_corporate_action_blocks() -> None:
        event = pd.DataFrame(
            [
                {
                    "Event_ID": "CA1",
                    "Symbol": "X",
                    "Event_Type": "SPLIT",
                    "Effective_Date": "2026-03-02",
                    "Payment_Date": None,
                    "Record_Date": None,
                    "Cash_Per_Share": None,
                    "Share_Ratio": 2.0,
                    "Reconciliation_Status": "BLOCKED_PENDING_OFFICIAL_RECONCILIATION",
                }
            ]
        )
        try:
            apply_corporate_actions("2026-03-02", 0, {"X": 10}, event, {})
        except PolicyBlockedError as error:
            assert error.reason == "CORPORATE_ACTION_NOT_RECONCILED"
            return
        raise AssertionError("Unreconciled action was applied.")

    case("32_unreconciled_corporate_action_blocked", assert_unreconciled_corporate_action_blocks)

    def assert_verified_split_applies_once() -> None:
        event = pd.DataFrame(
            [
                {
                    "Event_ID": "CA2",
                    "Symbol": "X",
                    "Event_Type": "SPLIT",
                    "Effective_Date": "2026-03-02",
                    "Payment_Date": None,
                    "Record_Date": None,
                    "Cash_Per_Share": None,
                    "Share_Ratio": 2.0,
                    "Reconciliation_Status": "VERIFIED_OFFICIAL_TERMS",
                }
            ]
        )
        result = apply_corporate_actions("2026-03-02", 5, {"X": 10}, event, {})
        repeated = apply_corporate_actions("2026-03-02", result["Cash"], result["Positions"], event, {}, result["Applied_Event_IDs"])
        assert result["Positions"]["X"] == 20 and repeated["Positions"]["X"] == 20

    case("33_verified_corporate_action_applied_once", assert_verified_split_applies_once)

    def assert_stale_threshold_and_unavailable_value() -> None:
        first = mark_portfolio("2026-03-01", 0, True, 0, {"X": 10}, {"X": {"Close": 5, "Status": "VERIFIED"}}, {})
        marks = first["Last_Valid_Marks"]
        statuses = []
        for ordinal in range(1, 5):
            valuation = mark_portfolio(
                pd.Timestamp("2026-03-01") + pd.Timedelta(days=ordinal),
                ordinal,
                True,
                0,
                {"X": 10},
                {},
                marks,
            )
            marks = valuation["Last_Valid_Marks"]
            statuses.append(valuation["Daily"]["Portfolio_Equity_Status"])
        assert statuses == [
            "PORTFOLIO_VALUE_STALE",
            "PORTFOLIO_VALUE_STALE",
            "PORTFOLIO_VALUE_STALE",
            "PORTFOLIO_VALUE_UNAVAILABLE",
        ]

    case("34_stale_marks_capped_at_three_sessions", assert_stale_threshold_and_unavailable_value)

    def assert_missing_close_without_mark_unavailable() -> None:
        result = mark_portfolio("2026-03-01", 0, True, 100, {"X": 10}, {}, {})
        assert result["Daily"]["Portfolio_Equity_Status"] == "PORTFOLIO_VALUE_UNAVAILABLE"
        assert np.isnan(result["Daily"]["Portfolio_Equity"])

    case("35_missing_close_without_prior_mark", assert_missing_close_without_mark_unavailable)

    def assert_unverified_calendar_cash_only_unavailable() -> None:
        result = mark_portfolio("2026-03-01", None, False, 100, {}, {}, {})
        assert result["Daily"]["Portfolio_Equity_Status"] == "PORTFOLIO_VALUE_UNAVAILABLE"

    case("36_calendar_required_for_daily_valuation", assert_unverified_calendar_cash_only_unavailable)

    def assert_segment_threshold_and_unresolved_rates() -> None:
        first_dates = pd.date_range("2020-01-01", periods=253, freq="D")
        second_dates = pd.date_range(first_dates[-1] + pd.Timedelta(days=2), periods=3, freq="D")
        rows = [
            {
                "Date": date,
                "Session_Ordinal": index,
                "Portfolio_Equity": 1000.0 * (1.001**index),
                "Portfolio_Equity_Status": "PORTFOLIO_VALUE_VALID",
                "Cash": 500.0,
                "Stale_Holding_Count": 0,
            }
            for index, date in enumerate(first_dates)
        ]
        rows.append(
            {
                "Date": first_dates[-1] + pd.Timedelta(days=1),
                "Session_Ordinal": 253,
                "Portfolio_Equity": np.nan,
                "Portfolio_Equity_Status": "PORTFOLIO_VALUE_STALE",
                "Cash": 500.0,
                "Stale_Holding_Count": 1,
            }
        )
        for index, date in enumerate(second_dates, start=254):
            rows.append(
                {
                    "Date": date,
                    "Session_Ordinal": index,
                    "Portfolio_Equity": 2000.0 + index,
                    "Portfolio_Equity_Status": "PORTFOLIO_VALUE_VALID",
                    "Cash": 1000.0,
                    "Stale_Holding_Count": 0,
                }
            )
        metrics = calculate_segment_metrics(pd.DataFrame(rows))
        assert len(metrics) == 2
        assert metrics.iloc[0]["Valid_Return_Intervals"] == 252
        assert metrics.iloc[0]["Annualization_Status"] == "ANNUALIZED_252_VERIFIED_INTERVALS"
        assert metrics.iloc[1]["Valid_Return_Intervals"] == 2
        assert metrics.iloc[1]["Annualization_Status"] == "UNANNUALIZED_BELOW_252_INTERVALS"
        assert metrics["Sharpe_Ratio"].isna().all()
        assert metrics["Sortino_Ratio"].isna().all()
        assert metrics.iloc[0]["Risk_Free_Status"] == "NEEDS_RESEARCHER_APPROVAL"
        assert metrics.iloc[0]["MAR_Status"] == "NEEDS_RESEARCHER_APPROVAL"

    case("37_no_bridging_252_threshold_rf_and_mar", assert_segment_threshold_and_unresolved_rates)
    return pd.DataFrame(results)


In [14]:
SYNTHETIC_TEST_RESULTS = run_synthetic_tests()
display(SYNTHETIC_TEST_RESULTS)
assert len(SYNTHETIC_TEST_RESULTS) == 37
assert SYNTHETIC_TEST_RESULTS['Status'].eq('PASS').all()
print('ผ่านการทดสอบสังเคราะห์ครบ 37 กรณี; ไม่มีการรัน backtest ย้อนหลัง')


,Case,Status,Details
0,01_membership_snapshot_scope,PASS,
1,02_membership_change_between_dates,PASS,
2,03_verified_membership_removal,PASS,
3,04_held_symbol_without_prediction,PASS,
4,05_unverified_membership_fails_closed,PASS,
5,06_approved_exclusion_blocks_buy,PASS,
6,07_security_identity_required,PASS,
7,08_corporate_action_reconciliation_required,PASS,
8,09_execution_price_eligibility_required,PASS,
9,10_prediction_execution_date_order,PASS,


ผ่านการทดสอบสังเคราะห์ครบ 37 กรณี; ไม่มีการรัน backtest ย้อนหลัง


In [15]:
# Targeted regression tests สำหรับข้อบกพร่องที่พบจากการตรวจ Notebook v2

def run_targeted_regression_tests() -> pd.DataFrame:
    results = []
    def check(name, fn):
        try:
            fn(); results.append({"Case": name, "Status": "PASS", "Details": ""})
        except Exception as exc:
            results.append({"Case": name, "Status": "FAIL", "Details": f"{type(exc).__name__}: {exc}"})

    def test_normal_always_hold():
        row = pd.DataFrame([{"Symbol":"AAA", "Prediction":2, "Probability_0":0.1, "Probability_1":0.1, "Probability_2":0.8}])
        for sid in STRATEGIES:
            out = strategy_signals(row, {"AAA":0.12}, sid, {})
            assert out.iloc[0]["Action"] == "HOLD"
            assert abs(out.iloc[0]["Target_Weight"] - 0.12) < 1e-12
    check("R01_normal_class_always_hold_all_strategies", test_normal_always_hold)

    def test_cartesian_matrix():
        expected = {
            "S1_PanicBuy_FOMOBuy": {0:"BUY",1:"BUY"},
            "S2_PanicBuy_FOMOSell": {0:"BUY",1:"SELL"},
            "S3_PanicBuy_FOMOHold": {0:"BUY",1:"HOLD"},
            "S4_PanicSell_FOMOBuy": {0:"SELL",1:"BUY"},
            "S5_PanicSell_FOMOSell": {0:"SELL",1:"SELL"},
            "S6_PanicSell_FOMOHold": {0:"SELL",1:"HOLD"},
            "S7_PanicHold_FOMOBuy": {0:"HOLD",1:"BUY"},
            "S8_PanicHold_FOMOSell": {0:"HOLD",1:"SELL"},
            "S9_PanicHold_FOMOHold": {0:"HOLD",1:"HOLD"},
        }
        for sid, actions in expected.items():
            for cls, action in actions.items():
                row = pd.DataFrame([{"Symbol":"AAA", "Prediction":cls}])
                assert strategy_signals(row, {}, sid, {}).iloc[0]["Action"] == action
    check("R02_cartesian_strategy_matrix", test_cartesian_matrix)

    def test_cvar_tied_boundary():
        losses = [10.0]*11 + [9.0]*3 + [0.0]*238
        dates = pd.date_range("2020-01-01", periods=252, freq="D")
        result = fractional_tail_cvar(losses, dates, 0.95)
        assert abs(result["Tail_Mass"] - 12.6) < 1e-12
        assert result["Strictly_Worse_Count"] == 11
        assert result["Tie_Group_Size"] == 3
        assert abs(result["Fractional_Weight_Per_Tied_Observation"] - (1.6/3)) < 1e-12
    check("R03_cvar_boundary_ties_share_fractionally", test_cvar_tied_boundary)

    def test_short_segment_sharpe_sortino_not_annualized():
        dates = pd.date_range("2025-01-01", periods=20, freq="B")
        equity = 1000.0 * np.cumprod(np.r_[1.0, np.repeat(1.001, 19)])
        daily = pd.DataFrame({"Date":dates, "Session_Ordinal":range(20), "Portfolio_Equity":equity,
                              "Portfolio_Equity_Status":"PORTFOLIO_VALUE_VALID", "Cash":500.0})
        out = calculate_segment_metrics(daily, risk_free_rate=0.02, risk_free_rate_approved=True,
                                        mar=0.0, mar_approved=True)
        assert out.iloc[0]["Valid_Return_Intervals"] == 19
        assert pd.isna(out.iloc[0]["Sharpe_Ratio"])
        assert pd.isna(out.iloc[0]["Sortino_Ratio"])
        assert out.iloc[0]["Risk_Free_Status"] == "UNANNUALIZED_BELOW_252_INTERVALS"
    check("R04_sharpe_sortino_obey_e004_252_intervals", test_short_segment_sharpe_sortino_not_annualized)

    def test_nonfinite_features_blocked():
        class Dummy:
            classes_ = np.array([0,1,2])
            def predict_proba(self, x):
                return np.tile([0.2,0.3,0.5], (len(x),1))
        row = pd.DataFrame([{**{name:0.1 for name in FEATURES}, "Symbol":"AAA"}])
        row.loc[0, FEATURES[0]] = np.nan
        lineage = {"Model_Version":"M1", "Prediction_Snapshot_ID":"P1"}
        try:
            generate_predictions(Dummy(), row, "2026-01-01", "2026-01-02", "VERIFIED", True, lineage)
        except PolicyBlockedError as exc:
            assert exc.reason == "FEATURE_VALUES_NONFINITE"
            return
        raise AssertionError("Non-finite feature was not blocked")
    check("R05_nonfinite_features_fail_closed", test_nonfinite_features_blocked)

    def test_performance_schema_complete():
        required = {"Average_Cash_Weight", "Transaction_Count", "Total_Transaction_Cost", "Total_Turnover",
                    "Stale_Valuation_Count", "Maximum_Stale_Verified_Sessions"}
        assert required.issubset(set(PERFORMANCE_COLUMNS))
    check("R06_performance_schema_contains_required_metrics", test_performance_schema_complete)
    return pd.DataFrame(results)

TARGETED_REGRESSION_RESULTS = run_targeted_regression_tests()
display(TARGETED_REGRESSION_RESULTS)
assert TARGETED_REGRESSION_RESULTS["Status"].eq("PASS").all()
print("Targeted regression tests ผ่านครบ; ไม่ได้รัน historical backtest")


,Case,Status,Details
0,R01_normal_class_always_hold_all_strategies,PASS,
1,R02_cartesian_strategy_matrix,PASS,
2,R03_cvar_boundary_ties_share_fractionally,PASS,
3,R04_sharpe_sortino_obey_e004_252_intervals,PASS,
4,R05_nonfinite_features_fail_closed,PASS,
5,R06_performance_schema_contains_required_metrics,PASS,


Targeted regression tests ผ่านครบ; ไม่ได้รัน historical backtest


## สถานะหลังแก้ไขแบบเจาะจง

ไฟล์นี้เป็นฉบับแก้ไขจาก v2 โดยปรับ Strategy Matrix ให้ตรงกับ 9 กลยุทธ์เดิม, Normal=Hold, CVaR tied-boundary fractional allocation, E-004 annualization gate สำหรับ Sharpe/Sortino, finite-value validation ก่อน inference และ Performance Summary schema เพิ่มเติม

ยังไม่ได้รัน historical backtest, ไม่ได้ fit/refit model, ไม่ได้ดาวน์โหลดข้อมูล และไม่เปลี่ยนสถานะ `RESEARCH_BACKTEST_GATE = BLOCKED` / `PHASE_F = DO_NOT_BEGIN`

เมื่อเปิดใน environment ของโครงการ ให้รันจากต้นจนจบเพื่อให้ 37 original synthetic tests และ 6 targeted regression tests ทำงานกับ dependency และ project paths จริง